# II-5 物理層・データリンク層の観測（ロジックアナライザ）（ノートブック版）

教科書の II-5 のうち、**ロジックアナライザ・Pico・Pi がなくても動く解析の部分**を、教科書の `synth_capture.py` が作る**合成の記録**（手順 A と同じもの）で試せるように1冊にまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。執筆時の Mac で全部で約1分）。

| 節 | 試すこと | 教科書の手順 | 使うファイル |
|---|---|---|---|
| 1 | 合成の記録を作り、「正解」を見る | 手順 A（mac_ii5.sh の 1)） | `synth_capture.py` |
| 2 | 自前のデコーダで I2C・UART・SPI・エンコーダを読む | 手順 A・4)・6) | `la_decode.py` |
| 3 | レジスタをデータシートと照らし合わせ、Pico の記録・sigrok の注釈と比べる | 手順 A・4) | `check_regs.py` |
| 4 | SCL の周期・ボーレート・SPI・PWM を区間推定する（量子化の扱い） | 4) | `bus_stats.py` |
| 5 | プルアップ抵抗を変えた記録から、容量と立ち上がり時間を上限・下限で言う | 5) | `rise_compare.py` |
| 6 | エンコーダの位相差・デューティ、続けて測った値の自己相関 | 6) | `bus_stats.py enc` |
| 7 | 1サンプルを各層で追い、3つの時計を対応づける | 7) | `trace_join.py` |

- 次のセルたちが、II-5-6 の Mac の解析のファイル6つと、入門3 のドライバ `bme280.py` を、教科書と同じフォルダの並び（`home-lab/partII/ii5/` と `home-lab/lib/pico/`）で、このノートブックの隣に `%%writefile` で書き出します。`synth_capture.py`・`trace_join.py` は `../../lib/pico/bme280.py`（入門3 の補正式）を読み込むので、この並びにしてあります。中身は教科書と同じです。変えたのは、図の日本語フォントの候補を足したこと（`plt.rcParams["font.family"]` の行）だけです。
- 各節では、ターミナルの `uv run python ○○.py …` の代わりに、`run("○○.py …")` で同じ引数のまま実行します（中では `python ○○.py …` を別のプロセスで動かします）。表示を見てから、`ok()` で教科書の手順 A と II-5-7 の「確かめること」「結果の例（合成データの値）」を、`synth_capture.py` の正解とも比べながら確かめます。
- 合成の記録は、本物の `sigrok-cli -O binary` と同じ形（1サンプル1バイト、ビット0 = D0 …）です。ロジックアナライザの水晶のずれ（−18 ppm）、Pico の水晶のずれ（+12 ppm）、RC の立ち上がり、手で回したエンコーダの速さのむら、わざと1バイトだけ化けさせた読み出しなどが入っています（`synth_capture.py` の先頭の説明）。
- **入れていないもの（実機が要る部分）**：配線、Pi の UART の設定（手順 B）、Pico への `bus_demo.py`・`trace_node.py` の書き込み、`sigrok-cli` での記録、Pi の `uart_stamp.py`・`tcpdump`・DB の書き出し、Grafana のパネル（`pi_ii5.sh`・`mac_ii5.sh` の 2) 以降の実物の部分）、PulseView で波形を見ること。Pico と Pi のプログラム（`bus_demo.py`・`trace_node.py`・`uart_stamp.py`）は、下に読むためだけに載せてあります（実行しません）。自分の記録（`data/bus.bin` など）を Colab の左のファイルの欄から `home-lab/partII/ii5/data/` に上げれば、同じ `run("check_regs.py data/bus.bin 4000000 …")` で解析できるはずですが、執筆時には確かめていません。
- ⚡ 実物の配線の注意（Pi の GPIO は 3.3V、ロジックアナライザの GND を必ずつなぐ、エンコーダを 5V で動かさないなど）は、教科書の II-5-4 を読んでください。このノートブックは計算だけです。
- 乱数の種は `synth_capture.py` の中で固定されているので、数値はほぼ同じになります（numpy・scipy の版で最後の桁が変わることがあります）。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には numpy・scipy・pandas・matplotlib が最初から入っている。版は要確認）
import importlib, subprocess, sys
for mod, pkg in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import numpy, scipy, pandas, matplotlib
print("OK numpy", numpy.__version__, "scipy", scipy.__version__, "pandas", pandas.__version__, "matplotlib", matplotlib.__version__)
print("（教科書は numpy 2.5・scipy 1.18・pandas 3.1・matplotlib 3.11 で確かめた）")

# 教科書と同じフォルダの並びを作る（やり直したときも、最初のフォルダに戻ってから作る）
import os
os.chdir(os.environ.setdefault("II5_NB_HOME", os.getcwd()))
os.makedirs("home-lab/partII/ii5", exist_ok=True)
os.makedirs("home-lab/lib/pico", exist_ok=True)
print("作業フォルダ:", os.path.abspath("home-lab/partII/ii5"))

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

## Pico と Pi の側（読むだけ。実行しません）

教科書の手順では、Pico A の `bus_demo.py` が 200 ms ごとに BME280・INA219 を I2C で読み、UART で Pi に1行送り、SPI の折り返しと PWM を出します。それをロジックアナライザ（4 MHz、D0〜D7）で記録します。Pi では `uart_stamp.py` が UART の行に受け取った時刻をつけます。最終節では `trace_node.py` が1秒に1回 BME280 を読み、MQTT で送ります。このノートブックの 1 節の `synth_capture.py` は、これらの記録と同じ形のファイルを作ります。

**Pico A（MicroPython）｜~/home-lab/partII/ii5/pico/bus_demo.py（Pico には :main.py）**（読むだけ）

```python
# II-5：ロジックアナライザで観測するための「バスの見本」を出し続けるプログラム（Pico A、MicroPython）
# 置き場所: Mac の ~/home-lab/partII/ii5/pico/bus_demo.py → Pico には :main.py としてコピー
# 準備: 入門3 のドライバ（:lib/bme280.py・:lib/ina219.py）が Pico に入っていること
# ピン（II-5-4 の表と同じ）:
#   I2C0  SDA=GP4（ピン6）・SCL=GP5（ピン7）  BME280・INA219（入門3 と同じ配線）
#   UART0 TX=GP0（ピン1）→ 1kΩ → Pi の GPIO15/RXD（ピン10）、RX=GP1（ピン2）← 1kΩ ← Pi の GPIO14/TXD（ピン8）
#         つなぐのは Pi の起動と Pico の USB のあと、外すのは Pico の USB を抜く・Pi を止める前（II-5-4）
#   SPI1  SCK=GP10（ピン14）、MOSI=GP11（ピン15）、MISO=GP12（ピン16）、CS=GP13（ピン17）
#         GP11 と GP12 をジャンパ線でつなぐ（ループバック：送ったものがそのまま返ってくる）
#   PWM   GP15（ピン20）1kHz・25%（第0章と同じ）
# 1周期（PERIOD_MS）ごとに：I2C で BME280 と INA219 のレジスタを読む → UART に「U,番号,UUUU」→ SPI で8バイト送受信
# USB シリアルには、Pico が自分で読んだ値を H・D 行で出す（serial_logger.py で CSV にして check_regs.py で照合）
import binascii
import struct
import time

from machine import I2C, PWM, SPI, UART, Pin

from bme280 import BME280
from ina219 import INA219

I2C_FREQ = 100_000       # 400_000 にすると高速モード（手順で切り替えて比べる）
UART_BAUD = 115200
SPI_BAUD = 500_000       # 150MHz を割り切れる値。ロジックアナライザ 4MHz で 1ビット 8サンプル
PERIOD_MS = 200
HEADER_EVERY = 25        # H 行（列名）をくり返す間隔（周期の数）
INA_ADDR = 0x40
COLS = "seq,tk_us,i2c_hz,chip_id,calib_hex,ina_cfg,ina_cal,spi_biterr,uart_acks,i2c_err"

i2c = I2C(0, sda=Pin(4), scl=Pin(5), freq=I2C_FREQ)
uart = UART(0, baudrate=UART_BAUD, tx=Pin(0), rx=Pin(1), bits=8, parity=None, stop=1)
spi = SPI(1, baudrate=SPI_BAUD, polarity=0, phase=0, bits=8, firstbit=SPI.MSB,
          sck=Pin(10), mosi=Pin(11), miso=Pin(12))
cs = Pin(13, Pin.OUT, value=1)
pwm = PWM(Pin(15))
pwm.freq(1000)
pwm.duty_u16(16384)      # 16384 / 65536 = 25%

print("I,bus_demo i2c=%d uart=%d spi=%d" % (I2C_FREQ, UART_BAUD, SPI_BAUD))
print("I,found", [hex(a) for a in i2c.scan()])
bme = BME280(i2c)        # ここでリセット・補正係数の読み込み・設定の書き込みが流れる（起動直後を記録すると見える）
try:
    INA219(i2c, addr=INA_ADDR)   # 設定 0x199F とキャリブレーション 0x1000 を書く（入門3 の既定）
    has_ina = True
except OSError:
    has_ina = False
    print("I,INA219 が見つからないので、BME280 だけで続けます")


def be16(b):
    return struct.unpack(">H", b)[0]


def main():
    seq, acks, i2c_err = 0, 0, 0
    rxbuf = b""
    t_next = time.ticks_ms()
    while True:
        t_next = time.ticks_add(t_next, PERIOD_MS)
        seq += 1
        tk = time.ticks_us()
        # ---- I2C：読むだけ（値は変えない）
        try:
            chip = i2c.readfrom_mem(bme.addr, 0xD0, 1)[0]
            cal = i2c.readfrom_mem(bme.addr, 0x88, 24)
            if has_ina:
                cfg = be16(i2c.readfrom_mem(INA_ADDR, 0x00, 2))
                calr = be16(i2c.readfrom_mem(INA_ADDR, 0x05, 2))
            else:
                cfg, calr = -1, -1
        except OSError:
            i2c_err += 1
            chip, cal, cfg, calr = -1, b"", -1, -1
        # ---- UART：0x55（'U'）はビットが 1,0,1,0… と交互になるので、ボーレートの推定に向く
        uart.write("U,%d,UUUU\n" % seq)
        if uart.any():
            rxbuf += uart.read()
            while b"\n" in rxbuf:
                ln, rxbuf = rxbuf.split(b"\n", 1)
                if ln.startswith(b"A,"):
                    acks += 1
            rxbuf = rxbuf[-64:]
        # ---- SPI：送ったものが MISO に返ってくるか
        tx = bytes([0x55, 0xAA, 0x00, 0xFF, 0x0F, 0xF0, seq & 0xFF, 0xA5])
        rx = bytearray(len(tx))
        cs(0)
        spi.write_readinto(tx, rx)
        cs(1)
        biterr = 0
        for x, y in zip(tx, rx):
            biterr += bin(x ^ y).count("1")
        # ---- USB シリアルに記録
        if seq % HEADER_EVERY == 1:
            print("H," + COLS)
        print("D,%d,%d,%d,%d,%s,%d,%d,%d,%d,%d" % (seq, tk, I2C_FREQ, chip, binascii.hexlify(cal).decode(), cfg, calr, biterr, acks, i2c_err))
        while time.ticks_diff(t_next, time.ticks_ms()) > 0:
            time.sleep_ms(1)


main()
```

**Pico A（MicroPython）｜~/home-lab/partII/ii5/pico/trace_node.py（Pico には :main.py）**（読むだけ）

```python
# II-5 最終節：1サンプルを各層で追うための端末（Pico A、MicroPython）
# 置き場所: Mac の ~/home-lab/partII/ii5/pico/trace_node.py → Pico には :main.py としてコピー
# 準備: :lib/bme280.py（入門3）、umqtt.simple（第0章）、:labconfig.py（II-4 の labconfig_a.py と同じもの）
# 動き（PERIOD_MS ごと）:
#   1. BME280 を強制モードで1回測らせる（I2C の書き込み。ロジックアナライザで見える）
#   2. 状態レジスタ（0xF3）の measuring ビットが 0 になるまで 1ms ごとに読む（変換の時間が見える）
#   3. tk_rd = ticks_us() を記録してから、0xF7〜0xFE の8バイトを読む（I2C）
#   4. 入門3 の補正式で温度・気圧・湿度に直す
#   5. UART に「T,番号,adc_t」を送る（Pi の uart_stamp.py が Pi の時刻を付ける。ロジックアナライザでも見える）
#   6. tk_pub = ticks_us() を記録して、MQTT（QoS 0）で lab/pico1-ii5/telemetry に送る（tcpdump・DB で見える）
import json
import time

import network
import rp2
from machine import I2C, UART, Pin
from umqtt.simple import MQTTClient

import labconfig as cfg
from bme280 import BME280

PERIOD_MS = 1000
DEV = cfg.DEVICE_ID + "-ii5"          # "pico1-ii5"
TOPIC = ("lab/%s/telemetry" % DEV).encode()
REG_STATUS, REG_CTRL_MEAS, REG_DATA = 0xF3, 0xF4, 0xF7

rp2.country("JP")
wlan = network.WLAN(network.STA_IF)
i2c = I2C(0, sda=Pin(4), scl=Pin(5), freq=100_000)
uart = UART(0, baudrate=115200, tx=Pin(0), rx=Pin(1))
# 平均回数を 1 にして変換を短くする（測定時間の式はデータシートの付録で要確認。この設定で約9ms の目安）
bme = BME280(i2c, osrs_t=1, osrs_p=1, osrs_h=1, iir=0)
CTRL = bme._ctrl_meas | 0x01         # 強制モード（入門3 のドライバが作った設定に、モードのビットを足す）


def wifi():
    wlan.active(True)
    try:
        wlan.config(pm=network.WLAN.PM_NONE)       # 省電力を切る（名前はファームの版で要確認。II-4 と同じ）
    except Exception as e:
        print("I,pm の設定に失敗:", e)
    if not wlan.isconnected():
        wlan.connect(cfg.WIFI_SSID, cfg.WIFI_PASS)
        t0 = time.ticks_ms()
        while not wlan.isconnected() and time.ticks_diff(time.ticks_ms(), t0) < 30_000:
            time.sleep_ms(250)
    print("I,Wi-Fi", wlan.isconnected(), wlan.ifconfig()[0])


def mqtt():
    c = MQTTClient(DEV, cfg.MQTT_HOST, port=getattr(cfg, "MQTT_PORT", 1883),
                   user=cfg.MQTT_USER, password=cfg.MQTT_PASS, keepalive=30)
    c.connect()
    print("I,MQTT connected", cfg.MQTT_HOST)
    return c


def measure():
    i2c.writeto_mem(bme.addr, REG_CTRL_MEAS, bytes([CTRL]))
    time.sleep_ms(1)
    for _ in range(100):
        if not i2c.readfrom_mem(bme.addr, REG_STATUS, 1)[0] & 0x08:
            break
        time.sleep_ms(1)
    tk_rd = time.ticks_us()
    d = i2c.readfrom_mem(bme.addr, REG_DATA, 8)
    adc_p = (d[0] << 12) | (d[1] << 4) | (d[2] >> 4)
    adc_t = (d[3] << 12) | (d[4] << 4) | (d[5] >> 4)
    adc_h = (d[6] << 8) | d[7]
    return tk_rd, adc_t, adc_p, adc_h


def main():
    wifi()
    c = None
    seq, nf = 0, 0
    t_next = time.ticks_ms()
    while True:
        t_next = time.ticks_add(t_next, PERIOD_MS)
        seq += 1
        tk_rd, adc_t, adc_p, adc_h = measure()
        t_c, p_pa, h = bme.compensate(adc_t, adc_p, adc_h if bme.has_humidity else None)
        uart.write("T,%d,%d\n" % (seq, adc_t))       # 送信バッファに入れた時点で戻る（行が流れ終わるのは約1ms後）
        tk_pub = time.ticks_us()
        msg = json.dumps({"dev": DEV, "seq": seq, "adc_t": adc_t, "adc_p": adc_p, "adc_h": adc_h,
                          "temp_c": t_c, "press_pa": p_pa, "hum_pct": h,
                          "tk_rd": tk_rd, "tk_pub": tk_pub, "nf": nf})
        try:
            if c is None:
                if not wlan.isconnected():
                    wifi()
                c = mqtt()
            c.publish(TOPIC, msg)
        except OSError as e:
            nf += 1
            print("I,send fail", seq, e)
            try:
                c.disconnect()
            except Exception:
                pass
            c = None
        if seq % 10 == 0:
            print("I,seq", seq, "temp_c", t_c, "nf", nf)
        while time.ticks_diff(t_next, time.ticks_ms()) > 0:
            time.sleep_ms(1)


main()
```

**Pi で実行｜~/home-lab/partII/ii5/uart_stamp.py**（読むだけ）

```python
#!/usr/bin/env python3
"""II-5：Raspberry Pi 5 の UART（GPIO14/15）で受けた行に、Pi の時刻を付けて CSV に保存する

置き場所: Mac の ~/home-lab/partII/ii5/uart_stamp.py → Pi の ~/ii5/uart_stamp.py にコピー
準備（Pi で1回）: ~/labenv/bin/pip install pyserial
使い方（Pi の ~/ii5 で）:
  ~/labenv/bin/python uart_stamp.py --seconds 70 --out uart_trace.csv
  ~/labenv/bin/python uart_stamp.py --seconds 40 --out uart_bus.csv --echo
  --port   ：既定 /dev/ttyAMA0（GPIO14/15 の UART0。名前は ls -l /dev/ttyAMA* /dev/serial* で要確認）
  --echo   ：「U,番号,…」の行に「A,番号」を送り返す（Pi の TX をロジックアナライザで見るため）
時刻は time.time()（Pi の時計。tcpdump と mqtt2pg の recv_unix と同じ時計）で、行の最後の改行を受け取った直後に読む。
"""
import argparse
import csv
import pathlib
import sys
import time

import serial


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--port", default="/dev/ttyAMA0")
    ap.add_argument("--baud", type=int, default=115200)
    ap.add_argument("--seconds", type=float, default=60)
    ap.add_argument("--out", required=True)
    ap.add_argument("--echo", action="store_true")
    a = ap.parse_args()
    out = pathlib.Path(a.out)
    if out.exists():
        sys.exit(f"{out} はすでにあります（上書きしないように止めました）")
    s = serial.Serial(a.port, a.baud, timeout=0.2)
    s.reset_input_buffer()
    n = part = 0
    buf = b""
    t_end = time.time() + a.seconds
    with open(out, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["pi_unix", "line"])
        try:
            while time.time() < t_end:
                chunk = s.read(s.in_waiting or 1)
                t = time.time()
                if not chunk:
                    continue
                buf += chunk
                while b"\n" in buf:
                    ln, buf = buf.split(b"\n", 1)
                    text = ln.decode("ascii", "replace").strip()
                    if not text:
                        continue
                    if not text[0].isalpha():
                        part += 1               # 途中から受け取った行（記録を始めた瞬間など）
                        continue
                    w.writerow([f"{t:.6f}", text])
                    n += 1
                    if a.echo and text.startswith("U,"):
                        s.write(("A,%s\n" % text.split(",")[1]).encode())
        except KeyboardInterrupt:
            print("中断しました")
    s.close()
    print(f"{out} に {n} 行（形の崩れた行 {part}）")
    if n == 0:
        print("1行も来ません：配線（Pico の TX→Pi の RX）、GND、UART の有効化、シリアルコンソールとの衝突を確認")


if __name__ == "__main__":
    main()
```

次のセルで入門3 のドライバ `bme280.py` を、続く6つのセルで II-5-6 の Mac の解析のファイルを書き出します（`check_regs.py`・`bus_stats.py`・`rise_compare.py`・`trace_join.py` は、同じフォルダの `la_decode.py` を読み込みます）。

In [ ]:
%%writefile home-lab/lib/pico/bme280.py
# BME280 / BMP280 ドライバ（MicroPython 用。補正式のテストのため CPython でも import できる）
# 置き場所: Mac の ~/home-lab/lib/pico/bme280.py が正本 → Pico には :lib/bme280.py としてコピーする
# 使い方:
#   from machine import I2C, Pin
#   from bme280 import BME280
#   i2c = I2C(0, sda=Pin(4), scl=Pin(5), freq=100_000)
#   s = BME280(i2c)              # アドレスは 0x76 → 0x77 の順に探す
#   t, p, h = s.read()           # 温度 [°C]、気圧 [Pa]、湿度 [%RH]（BMP280 なら h は None）
#   print(s.name, s.chip_id)     # "BME280" と 0x60、または "BMP280" と 0x58
# 補正式は Bosch のデータシートにある整数版の式をそのまま書いたもの（式の番号・節はデータシートで要確認）。
import struct
import time

REG_CALIB_TP = 0x88      # 0x88〜0x9F：温度・気圧の補正係数（BME280・BMP280 共通）
REG_CALIB_H1 = 0xA1      # 湿度の補正係数（BME280 だけ）
REG_CALIB_H2 = 0xE1      # 0xE1〜0xE7
REG_ID = 0xD0
REG_RESET = 0xE0
REG_CTRL_HUM = 0xF2      # BME280 だけ
REG_STATUS = 0xF3
REG_CTRL_MEAS = 0xF4
REG_CONFIG = 0xF5
REG_DATA = 0xF7          # 0xF7〜0xFE（BMP280 は 0xF7〜0xFC）

CHIP_BME280 = 0x60
CHIP_BMP280 = (0x56, 0x57, 0x58)     # 0x58 が量産品。0x56・0x57 は試作品の値とされる（要確認）
OSRS_CODE = {0: 0, 1: 1, 2: 2, 4: 3, 8: 4, 16: 5}   # 平均回数（オーバーサンプリング）→ レジスタの値
IIR_CODE = {0: 0, 2: 1, 4: 2, 8: 3, 16: 4}


def _sleep_ms(ms):
    f = getattr(time, "sleep_ms", None)
    if f:
        f(ms)
    else:
        time.sleep(ms / 1000)


class BME280:
    def __init__(self, i2c, addr=None, osrs_t=2, osrs_p=16, osrs_h=1, iir=0):
        self.i2c = i2c
        if addr is None:
            found = i2c.scan()
            for a in (0x76, 0x77):
                if a in found:
                    addr = a
                    break
            else:
                raise OSError("BME280/BMP280 が 0x76・0x77 に見つかりません")
        self.addr = addr
        self.chip_id = self._r8(REG_ID)
        if self.chip_id == CHIP_BME280:
            self.name, self.has_humidity = "BME280", True
        elif self.chip_id in CHIP_BMP280:
            self.name, self.has_humidity = "BMP280", False
        else:
            raise OSError("チップID 0x%02X は BME280（0x60）でも BMP280（0x58）でもありません" % self.chip_id)
        self._w8(REG_RESET, 0xB6)              # ソフトリセット
        _sleep_ms(5)
        for _ in range(20):                    # 補正係数の読み込み（im_update）が終わるまで待つ
            if not self._r8(REG_STATUS) & 0x01:
                break
            _sleep_ms(2)
        self._read_calibration()
        self.configure(osrs_t, osrs_p, osrs_h, iir)

    # ---------------------------------------------------------------- レジスタの読み書き
    def _r8(self, reg):
        return self.i2c.readfrom_mem(self.addr, reg, 1)[0]

    def _w8(self, reg, val):
        self.i2c.writeto_mem(self.addr, reg, bytes([val & 0xFF]))

    def _read_calibration(self):
        c = self.i2c.readfrom_mem(self.addr, REG_CALIB_TP, 24)
        (self.T1, self.T2, self.T3,
         self.P1, self.P2, self.P3, self.P4, self.P5,
         self.P6, self.P7, self.P8, self.P9) = struct.unpack("<HhhHhhhhhhhh", c)
        if self.has_humidity:
            self.H1 = self._r8(REG_CALIB_H1)
            e = self.i2c.readfrom_mem(self.addr, REG_CALIB_H2, 7)
            self.H2 = struct.unpack("<h", e[0:2])[0]
            self.H3 = e[2]
            h4 = (e[3] << 4) | (e[4] & 0x0F)          # 12ビットの符号つき整数
            h5 = (e[5] << 4) | (e[4] >> 4)
            self.H4 = h4 - 4096 if h4 & 0x800 else h4
            self.H5 = h5 - 4096 if h5 & 0x800 else h5
            self.H6 = struct.unpack("<b", e[6:7])[0]

    def configure(self, osrs_t=2, osrs_p=16, osrs_h=1, iir=0):
        """平均回数（1, 2, 4, 8, 16。0 は測らない）と IIR フィルタ（0 = なし）を設定する"""
        self._ctrl_meas = (OSRS_CODE[osrs_t] << 5) | (OSRS_CODE[osrs_p] << 2)   # 最後の2ビット = モード
        if self.has_humidity:
            self._w8(REG_CTRL_HUM, OSRS_CODE[osrs_h])     # ctrl_meas を書いたときに有効になる
        self._w8(REG_CONFIG, IIR_CODE[iir] << 2)
        self._w8(REG_CTRL_MEAS, self._ctrl_meas)          # スリープモード

    # ---------------------------------------------------------------- 測定
    def read_raw(self):
        """強制モードで1回測り、補正前の値（adc_T, adc_P, adc_H）を返す"""
        self._w8(REG_CTRL_MEAS, self._ctrl_meas | 0x01)   # 強制モード：1回測ってスリープに戻る
        _sleep_ms(2)
        for _ in range(100):                               # measuring ビットが 0 になるまで待つ
            if not self._r8(REG_STATUS) & 0x08:
                break
            _sleep_ms(1)
        n = 8 if self.has_humidity else 6
        d = self.i2c.readfrom_mem(self.addr, REG_DATA, n)
        adc_p = (d[0] << 12) | (d[1] << 4) | (d[2] >> 4)
        adc_t = (d[3] << 12) | (d[4] << 4) | (d[5] >> 4)
        adc_h = (d[6] << 8) | d[7] if self.has_humidity else None
        return adc_t, adc_p, adc_h

    def read(self):
        """(温度 [°C], 気圧 [Pa], 湿度 [%RH] または None) を返す"""
        return self.compensate(*self.read_raw())

    # ---------------------------------------------------------------- 補正式（データシートの整数版）
    def compensate(self, adc_t, adc_p, adc_h=None):
        # 温度：t_fine は気圧・湿度の補正にも使う
        var1 = (((adc_t >> 3) - (self.T1 << 1)) * self.T2) >> 11
        var2 = (((((adc_t >> 4) - self.T1) * ((adc_t >> 4) - self.T1)) >> 12) * self.T3) >> 14
        t_fine = var1 + var2
        t_c = ((t_fine * 5 + 128) >> 8) / 100.0            # 0.01 °C 単位 → °C

        # 気圧（64ビット版。Python の整数は桁あふれしない）
        p_pa = None
        if adc_p != 0x80000:                               # 0x80000 は「測っていない」
            var1 = t_fine - 128000
            var2 = var1 * var1 * self.P6
            var2 = var2 + ((var1 * self.P5) << 17)
            var2 = var2 + (self.P4 << 35)
            var1 = ((var1 * var1 * self.P3) >> 8) + ((var1 * self.P2) << 12)
            var1 = (((1 << 47) + var1) * self.P1) >> 33
            if var1 != 0:
                p = 1048576 - adc_p
                p = (((p << 31) - var2) * 3125) // var1
                var1 = (self.P9 * (p >> 13) * (p >> 13)) >> 25
                var2 = (self.P8 * p) >> 19
                p = ((p + var1 + var2) >> 8) + (self.P7 << 4)
                p_pa = p / 256.0                            # Q24.8 形式 → Pa

        # 湿度（32ビット版）。BMP280 には湿度センサがない
        h_pct = None
        if self.has_humidity and adc_h is not None and adc_h != 0x8000:
            v = t_fine - 76800
            v = (((((adc_h << 14) - (self.H4 << 20) - (self.H5 * v)) + 16384) >> 15) *
                 (((((((v * self.H6) >> 10) * (((v * self.H3) >> 11) + 32768)) >> 10) + 2097152) *
                   self.H2 + 8192) >> 14))
            v = v - (((((v >> 15) * (v >> 15)) >> 7) * self.H1) >> 4)
            v = 0 if v < 0 else v
            v = 419430400 if v > 419430400 else v
            h_pct = (v >> 12) / 1024.0
        return t_c, p_pa, h_pct

In [ ]:
%%writefile home-lab/partII/ii5/la_decode.py
#!/usr/bin/env python3
"""II-5：sigrok-cli の binary 出力（1サンプル = 1バイト、ビット k = チャンネル Dk）を自前でデコードする

置き場所: ~/home-lab/partII/ii5/la_decode.py
使い方（~/home-lab/partII/ii5 で。2つ目の数はサンプリング周波数 [Hz]、sigrok-cli の samplerate と同じにする）:
  uv run python la_decode.py i2c  data/bus.bin 4000000 [--sda 0 --scl 1] [--max 20]
  uv run python la_decode.py uart data/bus.bin 4000000 --ch 2 [--baud 115200]
  uv run python la_decode.py spi  data/bus.bin 4000000 [--sck 4 --mosi 5 --cs 6]
  uv run python la_decode.py pwm  data/bus.bin 4000000 --ch 7
  uv run python la_decode.py enc  data/enc.bin 2000000 [--a 0 --b 1]
  uv run python la_decode.py ann  data/bus_i2c.txt        （sigrok-cli の I2C デコーダの出力を読んで表示）
check_regs.py・bus_stats.py・rise_compare.py・trace_join.py からも import して使う。

しくみ：記録の全サンプルを持ち歩かず、「どれかのチャンネルが変わったサンプルの番号と、そのときの8ビットの値」
だけを取り出してからデコードする（数千万サンプルの記録でもメモリが足りる）。
"""
import argparse
import re
import sys

import numpy as np

CHUNK = 1 << 24          # 一度に読むサンプル数（16M サンプル = 16MB）


class Capture:
    """sigrok-cli -O binary のファイル。変化点だけを持つ"""

    def __init__(self, path, fs):
        raw = np.memmap(path, dtype=np.uint8, mode="r")
        self.n = len(raw)
        self.fs = float(fs)
        if self.n < 2:
            sys.exit(f"{path} が空か短すぎます（記録に失敗していないか確認してください）")
        pos = [np.array([0], dtype=np.int64)]
        for s in range(0, self.n, CHUNK):
            seg = np.asarray(raw[s:min(self.n, s + CHUNK + 1)])
            pos.append(np.flatnonzero(seg[1:] != seg[:-1]).astype(np.int64) + 1 + s)
        self.idx = np.concatenate(pos)              # 変化したサンプルの番号（先頭の 0 を含む）
        self.val = np.asarray(raw[self.idx])         # そのときの8チャンネルの値
        self._edge = {}

    def edges(self, ch):
        """チャンネル ch の (変化したサンプル番号, 変化後の値, 最初の値)"""
        if ch not in self._edge:
            b = ((self.val >> ch) & 1).astype(np.int8)
            k = np.flatnonzero(np.diff(b)) + 1
            self._edge[ch] = (self.idx[k], b[k], int(b[0]))
        return self._edge[ch]

    def level(self, ch, s):
        """サンプル番号 s（配列可）でのチャンネル ch の値（0/1）"""
        pos, _, init = self.edges(ch)
        k = np.searchsorted(pos, s, side="right")
        return init ^ (k & 1)

    def seconds(self):
        return self.n / self.fs


# ------------------------------------------------------------------------------------------- I2C
def decode_i2c(cap, sda=0, scl=1):
    """I2C を読む。戻り値は「START〜STOP」ごとの辞書のリスト
    segs：START（反復スタートを含む）ごとの {addr, rw(0=書く,1=読む), data, acks, t(各バイトの最初のビットの位置), anack}
    scl_rise・scl_fall：その間の SCL の立ち上がり・立ち下がりのサンプル番号（タイミングの統計に使う）
    """
    m = (1 << sda) | (1 << scl)
    v = cap.val & m
    keep = np.concatenate([[True], v[1:] != v[:-1]])
    idx, v = cap.idx[keep], v[keep]
    d = ((v >> sda) & 1).astype(np.int8)
    c = ((v >> scl) & 1).astype(np.int8)
    trans, cur, seg, bits = [], None, None, []
    warn = 0

    def close_seg():
        nonlocal seg
        if seg is not None and cur is not None:
            seg["partial_bits"] = len(bits)
            cur["segs"].append(seg)
        seg = None

    for i in range(1, len(idx)):
        d0, c0, d1, c1 = d[i - 1], c[i - 1], d[i], c[i]
        s = int(idx[i])
        if c0 == 1 and c1 == 1 and d0 != d1:            # SCL が 1 の間に SDA が変わった
            if d1 == 0:                                  # START（または反復スタート）
                if cur is None:
                    cur = {"start": s, "stop": None, "segs": [], "scl_rise": [], "scl_fall": []}
                else:
                    close_seg()
                seg, bits = {"addr": None, "rw": None, "data": [], "acks": [], "t": [], "anack": False}, []
            else:                                        # STOP
                if cur is not None:
                    close_seg()
                    cur["stop"] = s
                    trans.append(cur)
                cur = None
            continue
        if c0 != c1 and d0 != d1:                        # 同じサンプルで両方変わった（遅いサンプリングで起こる）
            warn += 1
        if cur is None:
            continue
        if c0 == 0 and c1 == 1:                          # SCL の立ち上がりで SDA を読む
            cur["scl_rise"].append(s)
            if seg is None:
                continue
            if not bits:
                seg["t"].append(s)
            bits.append(int(d1))
            if len(bits) == 9:
                byte = 0
                for b in bits[:8]:                       # MSB から
                    byte = (byte << 1) | b
                ack = bits[8] == 0
                if seg["addr"] is None:
                    seg["addr"], seg["rw"] = byte >> 1, byte & 1
                    seg["anack"] = not ack
                    seg["t"].pop()                       # アドレスのバイトの位置は data に含めない
                    seg["t_addr"] = s
                else:
                    seg["data"].append(byte)
                    seg["acks"].append(ack)
                bits = []
        elif c0 == 1 and c1 == 0:
            cur["scl_fall"].append(s)
    if cur is not None:                                  # STOP が来ないまま記録が終わった
        close_seg()
        trans.append(cur)
    decode_i2c.warnings = warn
    return trans


def i2c_ops(trans):
    """START〜STOP の並びを「レジスタの読み書き」に直す
    ("read", addr, reg, data, t) ／ ("write", addr, reg, data, t) ／ ("nack", addr, None, [], t) ／ ("raw", …)
    t は START のサンプル番号。読み出しは「番地を書く → (反復)スタート → 読む」の形を1つにまとめる
    """
    ops = []
    pend = None                       # 番地だけを書いて STOP した場合（あとの読み出しと組にする）
    for tr in trans:
        segs = tr["segs"]
        j = 0
        while j < len(segs):
            sg = segs[j]
            if sg["addr"] is None:
                j += 1
                continue
            if sg["anack"]:
                ops.append(("nack", sg["addr"], None, [], tr["start"]))
                pend = None
                j += 1
                continue
            if sg["rw"] == 0:
                nxt = segs[j + 1] if j + 1 < len(segs) else None
                if len(sg["data"]) == 1 and nxt is not None and nxt["rw"] == 1 and nxt["addr"] == sg["addr"]:
                    ops.append(("read", sg["addr"], sg["data"][0], nxt["data"], tr["start"]))
                    j += 2
                    pend = None
                    continue
                if len(sg["data"]) == 1 and nxt is None:
                    pend = (sg["addr"], sg["data"][0], tr["start"])
                    j += 1
                    continue
                reg = sg["data"][0] if sg["data"] else None
                ops.append(("write", sg["addr"], reg, sg["data"][1:], tr["start"]))
                pend = None
            else:
                if pend is not None and pend[0] == sg["addr"]:
                    ops.append(("read", sg["addr"], pend[1], sg["data"], pend[2]))
                else:
                    ops.append(("raw", sg["addr"], None, sg["data"], tr["start"]))
                pend = None
            j += 1
    return ops


# ------------------------------------------------------------------------------------------ UART
def decode_uart(cap, ch, baud=115200):
    """8N1・LSB から・待機は 1。各フレームの {start, byte, ferr(ストップビットが 0), rel(フレーム内のエッジの位置)}"""
    pos, newv, _ = cap.edges(ch)
    T = cap.fs / baud
    falls = pos[newv == 0]
    frames = []
    nxt = -1.0
    for s in falls:
        if s < nxt:
            continue
        pts = s + (np.arange(10) + 0.5) * T
        if pts[-1] >= cap.n:
            break
        lv = cap.level(ch, np.floor(pts).astype(np.int64))
        if lv[0] != 0:                       # 細いひげ（スタートビットの途中で 1 に戻った）
            nxt = s + 1
            continue
        byte = 0
        for k in range(8):
            byte |= int(lv[1 + k]) << k
        lo, hi = np.searchsorted(pos, [s, s + 9.5 * T])
        rel = (pos[lo:hi] - s).astype(float)
        frames.append({"start": int(s), "byte": byte, "ferr": lv[9] == 0, "rel": rel})
        nxt = s + 9.5 * T
    return frames


def uart_lines(frames, fs, baud=115200):
    """フレームを改行（0x0A）で区切って行にする。(最初のスタートビットの位置, ストップビットの終わりの位置, 文字列, ferr の数)"""
    T = fs / baud
    out, buf, t0, nerr = [], [], None, 0
    for f in frames:
        if t0 is None:
            t0 = f["start"]
        nerr += f["ferr"]
        if f["byte"] == 0x0A:
            out.append((t0, f["start"] + 10 * T, bytes(buf).decode("ascii", "replace").rstrip("\r"), nerr))
            buf, t0, nerr = [], None, 0
        else:
            buf.append(f["byte"])
    return out


# ------------------------------------------------------------------------------------------- SPI
def decode_spi(cap, sck=4, mosi=5, miso=None, cs=6):
    """モード0（待機時 SCK=0、立ち上がりで読む）・MSB から・CS は 0 で有効。CS が 0 の区間ごとの辞書
    miso=None なら MISO は読まない（この章の折り返しでは MOSI と MISO が同じ線なので、比べても意味がない）"""
    cpos, cval, cinit = cap.edges(cs)
    starts = cpos[cval == 0]
    ends = cpos[cval == 1]
    if cinit == 0:
        starts = np.concatenate([[0], starts])
    spos, sval, _ = cap.edges(sck)
    rises = spos[sval == 1]
    frames = []
    for a in starts:
        k = np.searchsorted(ends, a, side="right")
        b = int(ends[k]) if k < len(ends) else cap.n
        r = rises[(rises > a) & (rises < b)]
        mo = cap.level(mosi, r)
        nb = len(r) // 8
        tx = [int("".join(str(x) for x in mo[8 * k:8 * k + 8]), 2) for k in range(nb)]
        rx = []
        if miso is not None:
            mi = cap.level(miso, r)
            rx = [int("".join(str(x) for x in mi[8 * k:8 * k + 8]), 2) for k in range(nb)]
        frames.append({"cs_fall": int(a), "cs_rise": int(b), "sck_rise": r, "mosi": tx, "miso": rx,
                       "extra_bits": len(r) - 8 * nb})
    return frames


# ------------------------------------------------------------------------------------------- PWM
def pwm_edges(cap, ch):
    """(立ち上がりの位置, それに続く立ち下がりの位置)。幅と周期はサンプル数で返す"""
    pos, val, _ = cap.edges(ch)
    rise = pos[val == 1]
    fall = pos[val == 0]
    fall = fall[fall > rise[0]] if len(rise) else fall
    n = min(len(rise), len(fall))
    return rise[:n], fall[:n]


# ------------------------------------------------------------------------------------- エンコーダ
# 状態 = (A<<1)|B。A が B より 90° 進む向き（10→11→01→00→10、図II-5-7）を +1（どちらが正転かは配線しだい）
QTAB = {(2, 3): 1, (3, 1): 1, (1, 0): 1, (0, 2): 1, (3, 2): -1, (1, 3): -1, (0, 1): -1, (2, 0): -1}


def decode_quad(cap, a=0, b=1):
    """4逓倍の直交デコード。(位置の時系列 [サンプル番号, カウント], 飛び（2段とび）の数)"""
    m = (1 << a) | (1 << b)
    v = cap.val & m
    keep = np.concatenate([[True], v[1:] != v[:-1]])
    idx, v = cap.idx[keep], v[keep]
    st = (((v >> a) & 1) << 1) | ((v >> b) & 1)
    cnt, jumps = 0, 0
    tpos = [(int(idx[0]), 0)]
    for i in range(1, len(st)):
        dlt = QTAB.get((int(st[i - 1]), int(st[i])))
        if dlt is None:
            jumps += 1                         # 00→11 など：サンプリングが遅すぎるか、ひげ
            continue
        cnt += dlt
        tpos.append((int(idx[i]), cnt))
    return np.array(tpos, dtype=np.int64), jumps


# ------------------------------------------------------------------- sigrok-cli の注釈の読み込み
# 例（sigrok-cli -P i2c:scl=D1:sda=D0 -A i2c=address-read:address-write:data-read:data-write）:
#   i2c-1: Address write: 76
#   i2c-1: Data write: D0
# --protocol-decoder-samplenum を付けると、行の先頭に「開始-終了 」のサンプル番号が付く（形は版で要確認）
ANN = re.compile(r'^\s*(?:(\d+)-(\d+)\s+)?([A-Za-z0-9_]+-\d+):\s+"?(.*?)"?\s*$')


def parse_ann(path):
    """[(開始サンプル or None, デコーダ名, 本文)]"""
    out = []
    with open(path, encoding="utf-8", errors="replace") as f:
        for ln in f:
            mm = ANN.match(ln)
            if mm:
                out.append((int(mm.group(1)) if mm.group(1) else None, mm.group(3), mm.group(4)))
    return out


def ann_i2c_ops(path):
    """sigrok-cli の I2C の注釈を、i2c_ops と同じ形 (種類, addr, reg, data, t) に直す"""
    segs = []
    for t, _, text in parse_ann(path):
        kind, _, val = text.partition(":")
        val = val.strip()
        if kind in ("Address write", "Address read"):
            segs.append({"addr": int(val, 16), "rw": 1 if kind.endswith("read") else 0, "data": [], "t": t})
        elif kind in ("Data write", "Data read") and segs:
            segs[-1]["data"].append(int(val, 16))
    ops, j = [], 0
    while j < len(segs):
        sg = segs[j]
        nxt = segs[j + 1] if j + 1 < len(segs) else None
        if sg["rw"] == 0 and len(sg["data"]) == 1 and nxt and nxt["rw"] == 1 and nxt["addr"] == sg["addr"]:
            ops.append(("read", sg["addr"], sg["data"][0], nxt["data"], sg["t"]))
            j += 2
            continue
        if sg["rw"] == 0:
            ops.append(("write", sg["addr"], sg["data"][0] if sg["data"] else None, sg["data"][1:], sg["t"]))
        else:
            ops.append(("raw", sg["addr"], None, sg["data"], sg["t"]))
        j += 1
    return ops


def hexs(b):
    return " ".join(f"{x:02X}" for x in b)


# ------------------------------------------------------------------------------------------- CLI
def main():
    ap = argparse.ArgumentParser(description="sigrok-cli の binary を自前でデコードする")
    sub = ap.add_subparsers(dest="cmd", required=True)
    for name in ("i2c", "uart", "spi", "pwm", "enc"):
        p = sub.add_parser(name)
        p.add_argument("file")
        p.add_argument("fs", type=float)
        p.add_argument("--max", type=int, default=20, help="表示する件数")
        if name == "i2c":
            p.add_argument("--sda", type=int, default=0)
            p.add_argument("--scl", type=int, default=1)
        if name == "uart":
            p.add_argument("--ch", type=int, default=2)
            p.add_argument("--baud", type=float, default=115200)
        if name == "spi":
            p.add_argument("--sck", type=int, default=4)
            p.add_argument("--mosi", type=int, default=5)
            p.add_argument("--cs", type=int, default=6)
        if name == "pwm":
            p.add_argument("--ch", type=int, default=7)
        if name == "enc":
            p.add_argument("--a", type=int, default=0)
            p.add_argument("--b", type=int, default=1)
    p = sub.add_parser("ann")
    p.add_argument("file")
    p.add_argument("--max", type=int, default=20)
    a = ap.parse_args()

    if a.cmd == "ann":
        ops = ann_i2c_ops(a.file)
        print(f"sigrok の注釈から {len(ops)} 件のレジスタ操作")
        for op in ops[:a.max]:
            print(f"  {op[0]:5s} 0x{op[1]:02X} reg={'--' if op[2] is None else f'0x{op[2]:02X}'} [{hexs(op[3])}]")
        return

    cap = Capture(a.file, a.fs)
    us = 1e6 / cap.fs
    print(f"{a.file}: {cap.n} サンプル（{cap.seconds():.3f} 秒）、1サンプル = {us:.4f} µs、変化点 {len(cap.idx)} 個")
    if a.cmd == "i2c":
        tr = decode_i2c(cap, a.sda, a.scl)
        ops = i2c_ops(tr)
        print(f"I2C：START〜STOP {len(tr)} 回、レジスタ操作 {len(ops)} 件、"
              f"SDA と SCL が同じサンプルで変わった箇所 {decode_i2c.warnings}")
        for op in ops[:a.max]:
            reg = "--" if op[2] is None else f"0x{op[2]:02X}"
            print(f"  t={op[4] / cap.fs * 1e3:10.3f} ms  {op[0]:5s} 0x{op[1]:02X} reg={reg} [{hexs(op[3])}]")
    elif a.cmd == "uart":
        fr = decode_uart(cap, a.ch, a.baud)
        ln = uart_lines(fr, cap.fs, a.baud)
        print(f"UART（D{a.ch}、{a.baud:.0f} baud）：{len(fr)} バイト、ストップビットの誤り {sum(f['ferr'] for f in fr)}、{len(ln)} 行")
        for t0, t1, s, ne in ln[:a.max]:
            print(f"  t={t0 / cap.fs * 1e3:10.3f} ms  {s!r}" + (f"  （誤り {ne}）" if ne else ""))
    elif a.cmd == "spi":
        fr = decode_spi(cap, a.sck, a.mosi, None, a.cs)
        # bus_demo.py が送るはずの並び：55 AA 00 FF 0F F0 seq A5（seq は1周期ごとに1増える）
        bad, seqs = 0, []
        for f in fr:
            m = f["mosi"]
            ok = len(m) == 8 and m[:6] == [0x55, 0xAA, 0x00, 0xFF, 0x0F, 0xF0] and m[7] == 0xA5
            bad += not ok
            if ok:
                seqs.append(m[6])
        jumps = sum((b - x) % 256 != 1 for x, b in zip(seqs, seqs[1:]))
        print(f"SPI：CS が 0 の区間 {len(fr)} 個、送るはずの並び（55 AA 00 FF 0F F0 seq A5）と違う区間 {bad} 個、"
              f"seq が1ずつ増えていない所 {jumps} 個")
        print("  折り返し（MISO に返ってきたか）は、この記録ではなく Pico の spi_biterr（bus_log.csv）で判定する")
        for f in fr[:a.max]:
            print(f"  t={f['cs_fall'] / cap.fs * 1e3:10.3f} ms  D5 [{hexs(f['mosi'])}]"
                  + (f"  余りビット {f['extra_bits']}" if f["extra_bits"] else ""))
    elif a.cmd == "pwm":
        r, f = pwm_edges(cap, a.ch)
        if len(r) < 3:
            sys.exit("立ち上がりが少なすぎます。配線・GND・サンプリング周波数を確認してください")
        w = (f - r) / cap.fs * 1e6
        T = np.diff(r) / cap.fs * 1e6
        print(f"PWM（D{a.ch}）：{len(r)} 周期、周期 {T.mean():.3f} µs（{1e6 / T.mean():.3f} Hz）、"
              f"幅 {w.mean():.3f} µs、デューティ {100 * w[:-1].sum() / T.sum():.3f} %")
        print(f"  幅のとる値（サンプル数）：{dict(zip(*np.unique(f - r, return_counts=True)))}")
    elif a.cmd == "enc":
        tp, jumps = decode_quad(cap, a.a, a.b)
        print(f"エンコーダ：エッジ {len(tp) - 1} 個、最後のカウント {tp[-1, 1]:+d}、2段とび {jumps} 回")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile home-lab/partII/ii5/check_regs.py
#!/usr/bin/env python3
"""II-5：ロジックアナライザで読んだ I2C のバイト列を、データシートのレジスタ定義と Pico の記録に照らし合わせる

置き場所: ~/home-lab/partII/ii5/check_regs.py
使い方: cd ~/home-lab/partII/ii5 && uv run python check_regs.py data/bus.bin 4000000 \
            --log data/bus_log.csv --ann data/bus_i2c.txt --calib-out data/calib.json
  bus.bin     ：sigrok-cli の binary（D0=SDA、D1=SCL、D2=Pico の TX）
  --log       ：bus_demo.py の D 行を serial_logger.py で保存した CSV（Pico が自分で読んだ値）
  --ann       ：sigrok-cli の I2C デコーダの出力（自前のデコードと sigrok のデコードを比べる）
  --calib-out ：BME280 の補正係数（T1〜P9）を JSON に保存する（trace_join.py で使う）

やること
  1. BME280：チップID（0xD0）、補正係数（0x88〜0x9F の24バイト）を読み、T1〜P9 に直す
  2. INA219：設定レジスタ（0x00）とキャリブレーション（0x05）のビットを読み、入門3 の ina219.py が書いた値と比べる
  3. UART の「U,番号,…」の行で、どの I2C の読み出しが Pico のどの周期のものかを決め、Pico の記録と1バイトずつ比べる
  4. 食い違いの率を区間推定する（読み出し1回を1単位にする。ビットを単位にするのは誤用）
"""
import argparse
import json
import math
import struct
import sys

import numpy as np
import pandas as pd
from scipy import stats

import la_decode as la

BME_ADDRS = (0x76, 0x77)
INA_ADDR = 0x40
CAL_NAMES = ["T1", "T2", "T3", "P1", "P2", "P3", "P4", "P5", "P6", "P7", "P8", "P9"]
# ina219.py の既定（range_mv=320, avg=1, bus_32v=False）なら設定は 0x199F、Cal = 4096 = 0x1000
INA_EXPECT_CFG, INA_EXPECT_CAL = 0x199F, 0x1000


def ina_fields(v):
    """INA219 の設定レジスタのビット（ビットの位置はデータシートで要確認。ina219.py と同じ読み方）"""
    return dict(RST=v >> 15 & 1, BRNG=v >> 13 & 1, PG=v >> 11 & 3, BADC=v >> 7 & 0xF, SADC=v >> 3 & 0xF, MODE=v & 7)


def wilson(x, n, z=1.96):
    if n == 0:
        return (math.nan, math.nan)
    p = x / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return max(0.0, c - h), min(1.0, c + h)


def clopper_pearson(x, n, conf=0.95):
    a = 1 - conf
    lo = stats.beta.ppf(a / 2, x, n - x + 1) if x > 0 else 0.0
    hi = stats.beta.ppf(1 - a / 2, x + 1, n - x) if x < n else 1.0
    return lo, hi


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("file")
    ap.add_argument("fs", type=float)
    ap.add_argument("--sda", type=int, default=0)
    ap.add_argument("--scl", type=int, default=1)
    ap.add_argument("--uart", type=int, default=2, help="Pico の TX をつないだチャンネル")
    ap.add_argument("--log", default=None)
    ap.add_argument("--ann", default=None)
    ap.add_argument("--calib-out", default=None)
    a = ap.parse_args()

    cap = la.Capture(a.file, a.fs)
    ops = la.i2c_ops(la.decode_i2c(cap, a.sda, a.scl))
    reads = [op for op in ops if op[0] == "read"]
    print(f"記録 {cap.seconds():.2f} 秒、I2C のレジスタ操作 {len(ops)} 件（読み出し {len(reads)} 件、"
          f"アドレスに返事なし {sum(op[0] == 'nack' for op in ops)} 件）")

    # ---- 1. BME280
    bme = [op for op in reads if op[1] in BME_ADDRS]
    ids = sorted({op[3][0] for op in bme if op[2] == 0xD0 and op[3]})
    print("\n[BME280]")
    for cid in ids:
        name = {0x60: "BME280", 0x58: "BMP280（量産品）", 0x56: "BMP280（試作）", 0x57: "BMP280（試作）"}.get(cid, "不明")
        print(f"  チップID 0x{cid:02X} → {name}")
    cals = [bytes(op[3]) for op in bme if op[2] == 0x88 and len(op[3]) == 24]
    calib = None
    if cals:
        vals, cnt = np.unique(np.array([list(c) for c in cals]), axis=0, return_counts=True)
        best = bytes(vals[np.argmax(cnt)].tolist())
        calib = dict(zip(CAL_NAMES, struct.unpack("<HhhHhhhhhhhh", best)))
        print(f"  補正係数の読み出し {len(cals)} 回、違う並びが {len(vals)} 種類（1種類なら毎回同じ）")
        print("  0x88〜0x9F:", la.hexs(best))
        print("  " + "  ".join(f"{k}={v}" for k, v in calib.items()))
        print("  読み方：2バイトずつ小さい桁が先（リトルエンディアン）。T1・P1 は符号なし、ほかは符号つき（データシートで要確認）")
        if len(vals) > 1:
            for v, c in zip(vals, cnt):
                print(f"    {c} 回：{la.hexs(v.tolist())}")
        if a.calib_out:
            with open(a.calib_out, "w") as f:
                json.dump(calib, f, indent=1)
            print(f"  → {a.calib_out} に保存しました")

    # ---- 2. INA219
    print("\n[INA219]")
    for reg, name, expect in [(0x00, "設定", INA_EXPECT_CFG), (0x05, "キャリブレーション", INA_EXPECT_CAL)]:
        vs = [(op[3][0] << 8) | op[3][1] for op in reads if op[1] == INA_ADDR and op[2] == reg and len(op[3]) == 2]
        if not vs:
            print(f"  0x{reg:02X}（{name}）の読み出しが記録にありません")
            continue
        u, c = np.unique(vs, return_counts=True)
        for v, k in zip(u, c):
            mark = "ina219.py の値と一致" if v == expect else f"ina219.py の値 0x{expect:04X} と違う"
            print(f"  0x{reg:02X}（{name}）= 0x{v:04X}  {k} 回  {mark}")
            if reg == 0x00:
                print("    ", ina_fields(int(v)), "（PG=3 → ±320mV、BADC・SADC=3 → 12ビット1回、MODE=7 → 連続）")
            else:
                print(f"     Cal = {v} → 電流の1目盛り = 0.04096 / (Cal × 0.1Ω) = {0.04096 / (v * 0.1) * 1e3:.4f} mA")
    writes = [op for op in ops if op[0] == "write"]
    if writes:
        print(f"  書き込みも {len(writes)} 件見えました（Pico の起動直後を記録したとき）:")
        for op in writes[:6]:
            print(f"    0x{op[1]:02X} reg=0x{op[2]:02X} [{la.hexs(op[3])}]")

    # ---- 3. Pico の記録と照合
    if a.log:
        log = pd.read_csv(a.log, dtype={"calib_hex": str})
        log = log[log["chip_id"] >= 0]          # Pico 側で I2C の失敗（chip_id = -1）だった周期は照合しない
        lines = la.uart_lines(la.decode_uart(cap, a.uart), cap.fs)
        marks = []
        for t0, _, s, _ in lines:
            p = s.split(",")
            if len(p) >= 2 and p[0] == "U" and p[1].isdigit():
                marks.append((t0, int(p[1])))
        print(f"\n[Pico の記録との照合] UART の U 行 {len(marks)} 本、Pico の記録 {len(log)} 行")
        bylog = log.set_index("seq")
        units, bad, nbits, badbits = 0, 0, 0, 0
        # 1周期の読み出しは、その周期の U 行の前にある。最初の U 行より前は途中から記録したかもしれないので使わない
        for (prev, _), (t0, seq) in zip(marks[:-1], marks[1:]):
            mine = [op for op in reads if prev <= op[4] < t0]
            if seq not in bylog.index:
                continue
            row = bylog.loc[seq]
            exp = {(0x76, 0xD0): [int(row.chip_id)],
                   (0x76, 0x88): list(bytes.fromhex(str(row.calib_hex))),
                   (0x40, 0x00): [int(row.ina_cfg) >> 8, int(row.ina_cfg) & 0xFF],
                   (0x40, 0x05): [int(row.ina_cal) >> 8, int(row.ina_cal) & 0xFF]}
            got = {}
            for op in mine:
                addr = 0x76 if op[1] in BME_ADDRS else op[1]
                got[(addr, op[2])] = op[3]
            for key, e in exp.items():
                units += 1
                g = got.get(key)
                nbits += 8 * len(e)
                if g is None or list(g) != e:
                    bad += 1
                    diff = 8 * len(e) if g is None or len(g) != len(e) else sum(bin(x ^ y).count("1") for x, y in zip(g, e))
                    badbits += diff
                    print(f"  食い違い：seq={seq} 0x{key[0]:02X} reg=0x{key[1]:02X}  LA [{la.hexs(g or [])}]  Pico [{la.hexs(e)}]")
        if units:
            lo, hi = clopper_pearson(bad, units)
            wl, wh = wilson(bad, units)
            print(f"  読み出し {units} 回のうち食い違い {bad} 回（{100 * bad / units:.3f} %）")
            print(f"    95% 区間：Clopper–Pearson {100 * lo:.3f}〜{100 * hi:.3f} %、Wilson {100 * wl:.3f}〜{100 * wh:.3f} %")
            if bad == 0:
                print(f"    0 回のときの目安（3の法則）：上限 ≈ 3/n = {300 / units:.3f} %")
            blo, bhi = clopper_pearson(badbits, nbits)
            print(f"  【誤用の例】ビット {nbits} 個を独立な試行とみなすと、誤り {badbits} ビット → 上限 {100 * bhi:.4f} %。")
            print("    区間が狭く見えるのは、同じ読み出しの中の8×数ビットを別々の試行と数えた（疑似反復）から。")
            print("    配線の接触不良や雑音は読み出し1回まるごと（または続けて何回も）を壊すので、単位は読み出し1回にする。")
        nz = int((log["spi_biterr"] != 0).sum())
        print(f"\n[SPI の折り返し] Pico の spi_biterr が 0 でない周期 {nz} / {len(log)}"
              "（0 でなければ GP11–GP12 のジャンパが外れている。LA の D5 は MOSI と MISO の同じ線なので、これは LA では判定できない）")

    # ---- 4. sigrok のデコーダと比べる
    if a.ann:
        sops = [op for op in la.ann_i2c_ops(a.ann) if op[0] in ("read", "write")]
        mops = [op for op in ops if op[0] in ("read", "write")]
        key = lambda op: (op[0], op[1], op[2], tuple(op[3]))  # noqa: E731
        same = sum(key(x) == key(y) for x, y in zip(sops, mops))
        print(f"\n[sigrok のデコーダとの比較] sigrok {len(sops)} 件、自前 {len(mops)} 件、先頭から順に比べて一致 {same} 件")
        if same != max(len(sops), len(mops)):
            for i, (x, y) in enumerate(zip(sops, mops)):
                if key(x) != key(y):
                    print(f"  最初の不一致 {i} 件目：sigrok {key(x)}  自前 {key(y)}")
                    break
            print("  記録の始まり・終わりで途中から／途中までのやりとりがあると、数が1〜2件ずれることがある")
    if not reads:
        sys.exit("I2C の読み出しが見つかりません。SDA・SCL のチャンネル、GND、サンプリング周波数を確認してください")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile home-lab/partII/ii5/bus_stats.py
#!/usr/bin/env python3
"""II-5：ロジックアナライザの記録から、バスのタイミングを推定する（区間つき）

置き場所: ~/home-lab/partII/ii5/bus_stats.py
使い方（~/home-lab/partII/ii5 で）:
  uv run python bus_stats.py i2c  data/bus.bin 4000000                 SCL の周期・High・Low（量子化の扱い）
  uv run python bus_stats.py uart data/bus.bin 4000000 --ch 2 3        ボーレートの誤差と許容範囲（D2=Pico、D3=Pi）
  uv run python bus_stats.py spi  data/bus.bin 4000000                 SCK の周波数
  uv run python bus_stats.py pwm  data/bus.bin 4000000 --ch 7          PWM の幅・周期と量子化のモデル
  uv run python bus_stats.py enc  data/enc.bin 2000000                 エンコーダの A・B 相の位相差（自己相関のある系列）
図は stats_<種類>.png に保存する。

考え方（II-5-3-6）
  ・エッジの時刻はサンプル間隔 1/fs の刻みでしか分からない（量子化）。周期が一定なら、1周期の測定値は
    m か m+1 サンプルの2通りだけになり、m+1 の割合が端数になる。平均は「最初と最後のエッジの差 ÷ 周期の数」と同じで、
    誤差は 1/(fs × 周期の数) 以下。
  ・続けて測った周期を独立とみなして sd/√N で区間を作るのは誤り（隣どうしの周期は端のエッジを共有し、
    手で回したエンコーダの速さはゆっくり変わる）。単位（やりとり1回、フレーム1個）ごとにまとめるか、
    有効サンプル数・ブロック・ブートストラップを使う。
"""
import argparse
import math

import numpy as np
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import la_decode as la

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False


def t_ci(x, conf=0.95):
    x = np.asarray(x, float)
    n = len(x)
    m = x.mean()
    if n < 2:
        return m, math.nan, math.nan
    h = stats.t.ppf(0.5 + conf / 2, n - 1) * x.std(ddof=1) / math.sqrt(n)
    return m, m - h, m + h


def regular(d, lo=0.8, hi=1.25):
    """中央値の lo〜hi 倍のものだけ残す（START・STOP・バイトの間のすき間を除く）"""
    d = np.asarray(d, float)
    if len(d) == 0:
        return d
    m = np.median(d)
    return d[(d > lo * m) & (d < hi * m)]


def quant_report(vals, label, fs):
    """サンプル数の整数値の分布と、「一定の周期を量子化した」モデルとの比較"""
    u, c = np.unique(vals.astype(int), return_counts=True)
    print(f"  {label} のとる値（サンプル数: 回数）:", ", ".join(f"{a}: {b}" for a, b in zip(u, c)))
    m = int(np.floor(vals.mean()))
    p = vals.mean() - m
    sd_q = math.sqrt(p * (1 - p))
    print(f"    平均 {vals.mean():.4f} サンプル → 量子化だけなら {m} と {m + 1} が {1 - p:.2f}:{p:.2f}、"
          f"標準偏差 {sd_q:.3f} サンプルのはず。実際は {vals.std(ddof=1):.3f} サンプル")
    if len(u) > 3 or vals.std(ddof=1) > sd_q + 0.3:
        print("    → 量子化だけでは説明できない広がり（本当のゆらぎ、クロックストレッチ、取りこぼし）がある")


def cmd_i2c(cap, a):
    tr = la.decode_i2c(cap, a.sda, a.scl)
    per_tr, highs, lows, allp = [], [], [], []
    for t in tr:
        r = np.array(t["scl_rise"])
        f = np.array(t["scl_fall"])
        if len(r) < 10:
            continue
        d = regular(np.diff(r))
        if len(d) < 5:
            continue
        per_tr.append(d.mean())
        allp.append(d)
        k = np.searchsorted(f, r)
        ok = k < len(f)
        highs.append(regular(f[k[ok]] - r[ok]))
        k2 = np.searchsorted(r, f)
        ok2 = k2 < len(r)
        lows.append(regular(r[k2[ok2]] - f[ok2]))
    if not per_tr:
        raise SystemExit("SCL の周期が取れません（I2C のやりとりが記録にないか、チャンネルが違う）")
    allp = np.concatenate(allp)
    us = 1e6 / cap.fs
    m, lo, hi = t_ci(per_tr)
    print(f"I2C：やりとり {len(per_tr)} 回、SCL の周期 {len(allp)} 個（1サンプル = {us * 1e3:.1f} ns）")
    print(f"  周期 = {m * us:.4f} µs（95% 区間 {lo * us:.4f}〜{hi * us:.4f}、やりとりを単位にした t 区間）"
          f" → SCL {1e3 / (m * us):.3f} kHz（{1e3 / (hi * us):.3f}〜{1e3 / (lo * us):.3f}）")
    naive = allp.std(ddof=1) / math.sqrt(len(allp))
    print(f"  【誤用の例】周期 {len(allp)} 個を独立とみなした標準誤差 {naive * us * 1e3:.3f} ns"
          f"（やりとり単位では {np.std(per_tr, ddof=1) / math.sqrt(len(per_tr)) * us * 1e3:.3f} ns）")
    print("    隣どうしの周期は端のエッジを共有する（量子化の誤差が負の相関をもつ）ので、sd/√N はあてにならない")
    quant_report(allp, "1周期", cap.fs)
    H, L = np.concatenate(highs), np.concatenate(lows)
    print(f"  High {H.mean() * us:.3f} µs、Low {L.mean() * us:.3f} µs（ロジックアナライザのしきい値で見た長さ）")
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    u, c = np.unique(allp.astype(int), return_counts=True)
    ax[0].bar(u, c, color="#4a6a90")
    ax[0].set_xlabel("SCL の1周期 [サンプル]")
    ax[0].set_ylabel("回数")
    ax[0].set_title("1周期の測定値は整数のサンプル数")
    ax[1].plot(np.array(per_tr) * us, ".", color="#4a6a90")
    ax[1].axhline(m * us, color="#c0504d")
    ax[1].set_xlabel("やりとりの番号")
    ax[1].set_ylabel("やりとりごとの平均周期 [µs]")
    ax[1].set_title("平均は 1 サンプルより細かく決まる")
    fig.tight_layout()
    fig.savefig("stats_i2c.png", dpi=120)


def uart_tolerance():
    """受信側が16倍で読み、ストップビットの中央（スタートの端から 9.5 ビット）で判定する場合の目安"""
    total = (0.5 - 1 / 16) / 9.5
    return total, total / 2


def cmd_uart(cap, a):
    total, each = uart_tolerance()
    rows = []
    fig, ax = plt.subplots(figsize=(7, 3.6))
    for ch in a.ch:
        fr = la.decode_uart(cap, ch, a.baud)
        T0 = cap.fs / a.baud
        slopes = []
        for f in fr:
            rel = f["rel"]
            if len(rel) < 3:
                continue
            k = np.round(rel / T0)
            if len(np.unique(k)) < 3 or np.max(np.abs(rel - k * T0)) > 0.3 * T0:
                continue
            slopes.append(np.polyfit(k, rel, 1)[0])
        if len(slopes) < 2:
            print(f"D{ch}：使えるフレームが足りません（0x55 = 'U' のようにエッジの多いバイトを送ると精度が上がる）")
            continue
        slopes = np.array(slopes)
        m, lo, hi = t_ci(slopes)
        baud, blo, bhi = cap.fs / m, cap.fs / hi, cap.fs / lo
        ppm = (baud / a.baud - 1) * 1e6
        rows.append((ch, baud, blo, bhi))
        print(f"D{ch}：フレーム {len(slopes)} 個、1ビット = {m / cap.fs * 1e6:.4f} µs → {baud:.1f} baud"
              f"（95% 区間 {blo:.1f}〜{bhi:.1f}）、{a.baud:.0f} からのずれ {ppm:+.0f} ppm")
        ax.hist((cap.fs / slopes / a.baud - 1) * 1e6, bins=30, alpha=0.6, label=f"D{ch}（フレームごと）")
    print(f"許容範囲の目安：両側のずれの合計で ±{100 * total:.2f} %（片側に半分ずつなら ±{100 * each:.2f} %）。"
          "立ち上がりの遅れや雑音があると狭くなる")
    print("注意：ここでの baud はロジックアナライザの水晶を基準にした値。水晶のずれ（数十 ppm、要確認）より小さい差は、"
          "Pico と ロジックアナライザのどちらのずれか区別できない（識別できない）")
    if len(rows) == 2:
        (c1, b1, _, _), (c2, b2, _, _) = rows
        print(f"D{c1} と D{c2} の差：{(b1 / b2 - 1) * 1e6:+.0f} ppm（ロジックアナライザのずれは比をとると消える）"
              f" → 許容範囲 ±{100 * total:.2f} % に対して十分小さいか")
    p_div = 8 * 150e6 / 115200
    ibrd, fbrd = int(p_div) >> 7, ((int(p_div) & 0x7F) + 1) // 2
    print(f"参考：Pico の UART のクロックを 150MHz とした計算値（要確認）＝ 150e6/(16×({ibrd}+{fbrd}/64)) = "
          f"{150e6 / (16 * (ibrd + fbrd / 64)):.1f} baud")
    ax.set_xlabel("ボーレートのずれ [ppm]（ロジックアナライザ基準）")
    ax.set_ylabel("フレーム数")
    ax.legend()
    fig.tight_layout()
    fig.savefig("stats_uart.png", dpi=120)


def cmd_spi(cap, a):
    fr = la.decode_spi(cap, a.sck, a.mosi, None, a.cs)
    per = [regular(np.diff(f["sck_rise"])).mean() for f in fr if len(f["sck_rise"]) > 8]
    m, lo, hi = t_ci(per)
    print(f"SPI：区間 {len(per)} 個、SCK = {cap.fs / m / 1e3:.3f} kHz（95% 区間 {cap.fs / hi / 1e3:.3f}〜{cap.fs / lo / 1e3:.3f}）")
    allp = np.concatenate([regular(np.diff(f["sck_rise"])) for f in fr if len(f["sck_rise"]) > 8])
    quant_report(allp, "SCK の1周期", cap.fs)


def cmd_pwm(cap, a):
    r, f = la.pwm_edges(cap, a.ch)
    w = (f - r).astype(float)
    T = np.diff(r).astype(float)
    us = 1e6 / cap.fs
    print(f"PWM（D{a.ch}）：{len(T)} 周期、周期 {T.mean() * us:.4f} µs、幅 {w.mean() * us:.4f} µs、"
          f"デューティ {100 * w[:-1].sum() / T.sum():.4f} %")
    quant_report(w, "幅", cap.fs)
    quant_report(T, "周期", cap.fs)
    tot = (r[-1] - r[0]) / cap.fs
    print(f"  周波数 = 周期の数 ÷ (最後と最初の立ち上がりの差) = {len(T) / tot:.4f} Hz、"
          f"量子化による誤差 ±{1 / tot / cap.fs * len(T) / tot:.5f} Hz 以下（ロジックアナライザの水晶のずれは別）")


def acf(x, nlag):
    x = np.asarray(x, float) - np.mean(x)
    d = np.dot(x, x)
    return np.array([np.dot(x[:len(x) - k], x[k:]) / d for k in range(nlag + 1)])


def block_boot(x, stat, L, B=2000, seed=0):
    rng = np.random.default_rng(seed)
    n = len(x)
    nb = int(np.ceil(n / L))
    out = np.empty(B)
    for b in range(B):
        st = rng.integers(0, n - L + 1, nb)
        out[b] = stat(np.concatenate([x[s:s + L] for s in st])[:n])
    return np.percentile(out, [2.5, 97.5])


def cmd_enc(cap, a):
    tp, jumps = la.decode_quad(cap, a.a, a.b)
    pa, va, _ = cap.edges(a.a)
    pb, vb, _ = cap.edges(a.b)
    ar, af, br = pa[va == 1], pa[va == 0], pb[vb == 1]
    pos_t, pos_c = tp[:, 0], tp[:, 1]
    ph, du, per, tm = [], [], [], []
    for i in range(len(ar) - 1):
        s, e = ar[i], ar[i + 1]
        bs = br[(br > s) & (br < e)]
        fs_ = af[(af > s) & (af < e)]
        k0, k1 = np.searchsorted(pos_t, [s, e])
        steps = np.diff(pos_c[k0:k1 + 1])
        if len(bs) != 1 or len(fs_) != 1 or len(steps) == 0 or not (np.all(steps > 0) or np.all(steps < 0)):
            continue                                   # 向きが変わった周期・ひげのある周期は除く
        ph.append((bs[0] - s) / (e - s) * 360)
        du.append((fs_[0] - s) / (e - s))
        per.append((e - s) / cap.fs)
        tm.append(s / cap.fs)
    ph, du, per, tm = map(np.array, (ph, du, per, tm))
    if len(ph) < 20:
        raise SystemExit("使える周期が少なすぎます（軸を回しながら記録したか、A・B のチャンネルを確認）")
    sign = "+" if pos_c[-1] >= 0 else "−"
    print(f"エンコーダ：4逓倍のカウント {pos_c[-1]:+d}（向き {sign}）、2段とび {jumps} 回、使えた A 相の周期 {len(ph)} 個")
    print(f"  A 相の周期 {1e3 * np.median(per):.3f} ms（中央値）、最短 {1e3 * per.min():.3f} ms"
          f" → 1周期あたり最短 {per.min() * cap.fs:.0f} サンプル")
    if ph.mean() > 180:
        print("  位相差が 180° を超えている → B が A より進んでいる向き（逆回転）として 360° から引いて読む")
        ph = 360 - ph
    for name, x in [("位相差 [°]", ph), ("A 相のデューティ", du), ("A 相の周期 [ms]", 1e3 * per)]:
        n = len(x)
        r1 = acf(x, 1)[1]
        neff = n * (1 - r1) / (1 + r1) if r1 > 0 else n
        m, lo, hi = t_ci(x)
        se_eff = x.std(ddof=1) / math.sqrt(max(neff, 2))
        L = max(5, int(round(n ** (1 / 3))) * 2)
        blo, bhi = block_boot(x, np.mean, L)
        print(f"  {name}: 平均 {m:.3f}、独立とみなした 95% 区間 {lo:.3f}〜{hi:.3f}")
        print(f"    1つずれの自己相関 ρ1 = {r1:.2f} → 有効サンプル数 n(1−ρ)/(1+ρ) ≈ {neff:.0f}（n = {n}）、"
              f"区間 ±{1.96 * se_eff:.3f}；ブロック・ブートストラップ（長さ {L}）{blo:.3f}〜{bhi:.3f}")
    print("  【誤用の例】A 相の周期（手で回す速さ）はゆっくり変わるので ρ1 が 1 に近い。独立とみなした区間は狭すぎる")
    print("    （AR(1) の有効サンプル数の式は目安。ρ が 1 に近いときはブロック・ブートストラップの区間も不安定になる）")
    print("  位相差が 90° からずれるのは、磁石とセンサの取り付け・デューティのずれのため（4逓倍の刻みが不等間隔になる）")
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
    ax[0].plot(tm, 1e3 * per, lw=0.8, color="#4a6a90")
    ax[0].set_xlabel("時刻 [s]")
    ax[0].set_ylabel("A 相の周期 [ms]")
    ax[0].set_title("手で回した速さのむら")
    ax[1].plot(tm, ph, ".", ms=2, color="#4a6a90")
    ax[1].axhline(90, color="#888", ls="--")
    ax[1].set_xlabel("時刻 [s]")
    ax[1].set_ylabel("B の A に対する遅れ [°]")
    ax[1].set_title("位相差")
    k = np.arange(31)
    ax[2].bar(k, acf(ph, 30), color="#4a6a90", label="位相差")
    ax[2].plot(k, acf(per, 30), "o-", ms=3, color="#c0504d", label="周期")
    ax[2].axhline(0, color="#888")
    ax[2].set_xlabel("ずれ（周期の数）")
    ax[2].set_ylabel("自己相関")
    ax[2].legend()
    fig.tight_layout()
    fig.savefig("stats_enc.png", dpi=120)


def main():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    for name in ("i2c", "uart", "spi", "pwm", "enc"):
        p = sub.add_parser(name)
        p.add_argument("file")
        p.add_argument("fs", type=float)
        if name == "i2c":
            p.add_argument("--sda", type=int, default=0)
            p.add_argument("--scl", type=int, default=1)
        if name == "uart":
            p.add_argument("--ch", type=int, nargs="+", default=[2, 3])
            p.add_argument("--baud", type=float, default=115200)
        if name == "spi":
            p.add_argument("--sck", type=int, default=4)
            p.add_argument("--mosi", type=int, default=5)
            p.add_argument("--cs", type=int, default=6)
        if name == "pwm":
            p.add_argument("--ch", type=int, default=7)
        if name == "enc":
            p.add_argument("--a", type=int, default=0)
            p.add_argument("--b", type=int, default=1)
    a = ap.parse_args()
    cap = la.Capture(a.file, a.fs)
    {"i2c": cmd_i2c, "uart": cmd_uart, "spi": cmd_spi, "pwm": cmd_pwm, "enc": cmd_enc}[a.cmd](cap, a)
    print(f"（図：stats_{a.cmd}.png）" if a.cmd in ("i2c", "uart", "enc") else "")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile home-lab/partII/ii5/rise_compare.py
#!/usr/bin/env python3
"""II-5：プルアップ抵抗を変えた I2C の記録から、立ち上がりの遅れを「上限と下限」で言う

置き場所: ~/home-lab/partII/ii5/rise_compare.py
使い方: cd ~/home-lab/partII/ii5 && uv run python rise_compare.py --fs 24000000 \
            data/rise_a.bin:9900 data/rise_b.bin:4950 data/rise_c.bin:2600
  「ファイル:プルアップ抵抗の実測値[Ω]」を2つ以上（並列になった合成の値。手順で DE-200A で測る）
  --vdd 3.3 --la-th 0.8 2.0 --pico-th 0.8 2.0 ：電源電圧と、しきい値がありうる範囲 [V]（データシートで要確認）

考え方（II-5-3-2）
  ロジックアナライザ（LA）は、線の電圧が自分のしきい値 V_LA を横切った時刻しか分からない。立ち上がりの形は見えない。
  ・Low の長さ（LA で見た立ち下がり → 立ち上がり）＝ Pico が数える Low の時間 ＋ RC·ln(1/(1 − V_LA/VDD))
  ・1周期 ＝ 一定の時間 ＋ RC·ln(1/(1 − V_P/VDD))（Pico は SCL が High になったと感じてから High を数える）
  R を変えて直線を当てはめると、傾き α = C·ln(1/(1−x_LA))、β = C·ln(1/(1−x_P))（x = しきい値/VDD）。
  しきい値はデータシートの範囲でしか分からないので、C と立ち上がり時間 t_r = 0.8473·R·C は
  「この範囲にある」としか言えない（部分的にしか識別できない）。それでも、規格の上限との比較には使える。
"""
import argparse
import math

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import la_decode as la
from bus_stats import regular

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False
K30_70 = math.log(0.7 / 0.3)          # = 0.8473（30% → 70% の立ち上がり時間 = 0.8473·RC）


def per_file(path, fs, sda, scl):
    cap = la.Capture(path, fs)
    lows, pers = [], []
    for t in la.decode_i2c(cap, sda, scl):
        r, f = np.array(t["scl_rise"]), np.array(t["scl_fall"])
        if len(r) < 10:
            continue
        k = np.searchsorted(r, f)
        ok = k < len(r)
        lo = regular(r[k[ok]] - f[ok])
        pe = regular(np.diff(r))
        if len(lo) > 5 and len(pe) > 5:
            lows.append(lo.mean() / fs)
            pers.append(pe.mean() / fs)
    lows, pers = np.array(lows), np.array(pers)
    n = len(lows)
    if n < 3:
        raise SystemExit(f"{path}：I2C のやりとりが少なすぎます")
    return dict(n=n, low=lows.mean(), low_se=lows.std(ddof=1) / math.sqrt(n),
                per=pers.mean(), per_se=pers.std(ddof=1) / math.sqrt(n))


def wls(x, y, se):
    """重みつき最小二乗（重み 1/se²）。傾き・切片とその標準誤差、適合度の χ²"""
    w = 1 / np.asarray(se) ** 2
    xm = np.sum(w * x) / w.sum()
    ym = np.sum(w * y) / w.sum()
    sxx = np.sum(w * (x - xm) ** 2)
    b = np.sum(w * (x - xm) * (y - ym)) / sxx
    a = ym - b * xm
    chi2 = float(np.sum(w * (y - a - b * x) ** 2))
    return a, b, math.sqrt(1 / sxx), chi2


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("files", nargs="+", help="ファイル:抵抗[Ω]")
    ap.add_argument("--fs", type=float, required=True)
    ap.add_argument("--sda", type=int, default=0)
    ap.add_argument("--scl", type=int, default=1)
    ap.add_argument("--vdd", type=float, default=3.3)
    ap.add_argument("--la-th", type=float, nargs=2, default=[0.8, 2.0])
    ap.add_argument("--pico-th", type=float, nargs=2, default=[0.8, 2.0])
    a = ap.parse_args()
    if len(a.files) < 2:
        raise SystemExit("抵抗の値を変えた記録が2つ以上必要です")

    R, res = [], []
    print(f"1サンプル = {1e9 / a.fs:.1f} ns。1本のエッジの位置はこの刻みでしか分からない（形はまったく見えない）")
    for s in a.files:
        path, _, r = s.rpartition(":")
        d = per_file(path, a.fs, a.sda, a.scl)
        R.append(float(r))
        res.append(d)
        print(f"  R = {float(r):7.0f} Ω：やりとり {d['n']:4d} 回、Low {d['low'] * 1e9:8.1f} ± {d['low_se'] * 1e9:.2f} ns、"
              f"周期 {d['per'] * 1e9:8.1f} ± {d['per_se'] * 1e9:.2f} ns → SCL {1e-3 / d['per']:.2f} kHz")
    R = np.array(R)
    low = np.array([d["low"] for d in res])
    low_se = np.array([max(d["low_se"], 1e-12) for d in res])
    per = np.array([d["per"] for d in res])
    per_se = np.array([max(d["per_se"], 1e-12) for d in res])
    a0, al, al_se, chi_l = wls(R, low, low_se)
    p0, be, be_se, chi_p = wls(R, per, per_se)
    print(f"\n傾き α（Low）= {al * 1e12:.2f} ± {al_se * 1e12:.2f} ps/Ω（1kΩ 増やすと {al * 1e12:.1f} ns 長くなる）")
    print(f"傾き β（周期）= {be * 1e12:.2f} ± {be_se * 1e12:.2f} ps/Ω")
    if len(R) > 2:
        print(f"直線からの外れ：χ²（Low）= {chi_l:.1f}、χ²（周期）= {chi_p:.1f}、自由度 {len(R) - 2}"
              "（自由度よりずっと大きいなら、抵抗の値の誤差や、RC だけでは説明できない遅れがある）")
    if be < 3 * be_se:
        print("周期が R でほとんど変わらない → 「Pico は High を感じてから数える」というモデルが合わない。"
              "このときは Low の傾き α だけを使う（RP2350 データシートの I2C の節で要確認）")

    def g(v):                         # しきい値 v [V] まで充電するのにかかる時間 ÷ RC
        return math.log(1 / (1 - v / a.vdd))
    # 2SE だけ広げて、統計のばらつきも含めた（保守的な）範囲にする
    c_from_a = ((al - 2 * al_se) / g(a.la_th[1]), (al + 2 * al_se) / g(a.la_th[0]))
    print(f"\nしきい値の範囲：LA {a.la_th[0]}〜{a.la_th[1]} V、Pico {a.pico_th[0]}〜{a.pico_th[1]} V（VDD = {a.vdd} V）")
    print(f"  α から：C = {c_from_a[0] * 1e12:.0f}〜{c_from_a[1] * 1e12:.0f} pF")
    lo_c, hi_c = c_from_a
    if be > 3 * be_se:
        c_from_b = ((be - 2 * be_se) / g(a.pico_th[1]), (be + 2 * be_se) / g(a.pico_th[0]))
        print(f"  β から：C = {c_from_b[0] * 1e12:.0f}〜{c_from_b[1] * 1e12:.0f} pF")
        lo_c, hi_c = max(lo_c, c_from_b[0]), min(hi_c, c_from_b[1])
        ratio = be / al
        print(f"  β/α = {ratio:.2f}（1 より大きければ、Pico のしきい値のほうが LA のしきい値より高い）")
    if lo_c > hi_c:
        print("  2つの範囲が重ならない → しきい値の範囲かモデルが間違っている")
        return
    print(f"  合わせると C = {lo_c * 1e12:.0f}〜{hi_c * 1e12:.0f} pF（LA のプローブの容量も含む）")
    print("\n30%→70% の立ち上がり時間 t_r = 0.8473·R·C の範囲（規格の上限：100kHz で 1000ns、400kHz で 300ns。規格書で要確認）")
    for r in R:
        tlo, thi = K30_70 * r * lo_c * 1e9, K30_70 * r * hi_c * 1e9
        verdict = []
        for name, lim in (("100kHz", 1000), ("400kHz", 300)):
            verdict.append(f"{name}：" + ("規格内と言える" if thi <= lim else "規格外と言える" if tlo > lim else "どちらとも言えない"))
        print(f"  R = {r:7.0f} Ω：{tlo:6.0f}〜{thi:6.0f} ns　" + "、".join(verdict))
    print("見えないもの：立ち上がりの形、30% や 70% を横切る時刻そのもの、リンギング。測るにはオシロスコープ相当が必要")

    fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
    xx = np.linspace(0, R.max() * 1.1, 50)
    for k, (y, se, c0, c1, name) in enumerate([(low, low_se, a0, al, "Low の長さ"), (per, per_se, p0, be, "SCL の周期")]):
        ax[k].errorbar(R / 1e3, y * 1e6, yerr=2 * se * 1e6, fmt="o", color="#4a6a90", capsize=4)
        ax[k].plot(xx / 1e3, (c0 + c1 * xx) * 1e6, color="#c0504d")
        ax[k].set_xlabel("プルアップ抵抗 R [kΩ]")
        ax[k].set_ylabel(f"{name} [µs]")
        ax[k].set_title(f"{name}（誤差棒は ±2SE）")
    fig.tight_layout()
    fig.savefig("rise_compare.png", dpi=120)
    print("（図：rise_compare.png）")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile home-lab/partII/ii5/trace_join.py
#!/usr/bin/env python3
"""II-5 最終節：1サンプルを「センサ → I2C → Pico → Wi-Fi → MQTT → DB → Grafana」の各層で観測した記録をつなぐ

置き場所: ~/home-lab/partII/ii5/trace_join.py
使い方: cd ~/home-lab/partII/ii5 && uv run python trace_join.py --la data/trace.bin --fs 2000000 \
            --uart data/uart_trace.csv --pcap data/mqtt_ii5.pcap --db data/db_ii5.csv \
            [--calib data/calib.json] [--refresh 5] [--seq 30]
  trace.bin      ：ロジックアナライザ（D0=SDA、D1=SCL、D2=Pico の TX → Pi の RX）
  uart_trace.csv ：Pi の uart_stamp.py が「T,番号,adc_t」の行に Pi の時刻を付けたもの
  mqtt_ii5.pcap  ：Pi の tcpdump（有線 eth0、ポート 1883）
  db_ii5.csv     ：PostgreSQL の messages 表から dev = 'pico1-ii5' の行を書き出したもの
  --calib        ：check_regs.py が保存した BME280 の補正係数（あれば、I2C のバイトから温度を計算し直して比べる）
  --refresh      ：Grafana のパネルの自動更新の間隔 [s]
  --seq          ：詳しく表示するサンプルの番号（省略すると、Pico → DB の遅れが中央値に近いもの）

時刻の対応づけ（II-5-3-7）
  ・Pi の上の記録（UART の受信、tcpdump、mqtt2pg の recv_unix）はすべて Pi の同じ時計 → そのまま引き算できる
  ・LA の時刻 → Pi の時刻：同じ UART の行を LA（ストップビットの終わり）と Pi（受け取った時刻）の両方で見ているので、
      Pi = a + b·LA ＋ 受信の遅れ（≥ 0）を、遅れの小さい行（下側 20%）で当てはめる。受信の遅れの最小値は分からないので、
      LA の出来事を Pi の時刻に直した値は「最大でその最小値ぶん遅め」に出る（片側の誤差）
  ・Pico の ticks_us → LA の時刻：Pico が I2C で読む直前に記録した tk_rd と、LA で見た START を同じように当てはめる
"""
import argparse
import csv
import json
import math
import pathlib
import re
import struct
import sys

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import la_decode as la

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False
DEV = "pico1-ii5"
TICKS_PERIOD = 1 << 30               # MicroPython の ticks_us は 2^30 µs（約17.9分）で一周する


# --------------------------------------------------------------------------------- 記録の読み込み
def read_pcap(path):
    """tcpdump -w の pcap（pcapng ではない）から (時刻, TCP の中身) を取り出す"""
    data = pathlib.Path(path).read_bytes()
    if data[:4] == b"\x0a\x0d\x0d\x0a":
        sys.exit("pcapng 形式です。tcpdump -w の既定の pcap で記録し直すか、変換してください")
    magic = data[:4]
    forms = {b"\xd4\xc3\xb2\xa1": ("<", 1e-6), b"\x4d\x3c\xb2\xa1": ("<", 1e-9),
             b"\xa1\xb2\xc3\xd4": (">", 1e-6), b"\xa1\xb2\x3c\x4d": (">", 1e-9)}
    if magic not in forms:
        sys.exit("pcap のファイルではありません")
    e, unit = forms[magic]
    link = struct.unpack(e + "I", data[20:24])[0]
    off, out = 24, []
    while off + 16 <= len(data):
        ts, tf, incl, _ = struct.unpack(e + "IIII", data[off:off + 16])
        p = data[off + 16:off + 16 + incl]
        off += 16 + incl
        if link == 1:                                   # Ethernet
            et, ip = struct.unpack(">H", p[12:14])[0], p[14:]
            if et == 0x8100:                            # VLAN
                et, ip = struct.unpack(">H", p[16:18])[0], p[18:]
        elif link == 113:                               # Linux cooked（-i any）
            et, ip = struct.unpack(">H", p[14:16])[0], p[16:]
        elif link == 276:                               # Linux cooked v2
            et, ip = struct.unpack(">H", p[0:2])[0], p[20:]
        else:
            continue
        if et != 0x0800 or len(ip) < 20 or ip[9] != 6:  # IPv4 の TCP だけ
            continue
        tcp = ip[(ip[0] & 15) * 4:]
        out.append((ts + tf * unit, tcp[(tcp[12] >> 4) * 4:]))
    return out


def pcap_by_seq(path):
    first, dup = {}, 0
    for t, pl in read_pcap(path):
        for m in re.finditer(rb"\{[^{}]*\}", pl):
            try:
                d = json.loads(m.group(0))
            except ValueError:
                continue
            if d.get("dev") != DEV or "seq" not in d:
                continue
            if d["seq"] in first:
                dup += 1                                # TCP の再送や QoS の送り直し：最初に見えた時刻を使う
            else:
                first[d["seq"]] = (t, d)
    return first, dup


def lower_fit(x, y, q=0.2, it=5):
    """y = a + b·x + 遅れ（≥0）を、遅れの小さい点で当てはめ、最後に「いちばん小さい遅れ = 0」になるよう a を決める"""
    x, y = np.asarray(x, float), np.asarray(y, float)
    x0 = x.mean()
    b, a = np.polyfit(x - x0, y, 1)
    for _ in range(it):
        r = y - (a + b * (x - x0))
        keep = r <= np.quantile(r, q)
        if keep.sum() < 3:
            break
        b, a = np.polyfit(x[keep] - x0, y[keep], 1)
    r = y - (a + b * (x - x0))
    a += r.min()
    r = r - r.min()
    return (lambda z: a + b * (np.asarray(z, float) - x0)), b, r


def unwrap_ticks(t):
    t = np.asarray(t, dtype=np.int64)
    d = np.diff(t) % TICKS_PERIOD
    return np.concatenate([[0], np.cumsum(d)]) + t[0]


def q(x, p):
    return float(np.percentile(x, p))


# ------------------------------------------------------------------------------------------- 本体
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--la", required=True)
    ap.add_argument("--fs", type=float, required=True)
    ap.add_argument("--uart", required=True)
    ap.add_argument("--pcap", required=True)
    ap.add_argument("--db", required=True)
    ap.add_argument("--calib", default=None)
    ap.add_argument("--refresh", type=float, default=5.0)
    ap.add_argument("--seq", type=int, default=None)
    a = ap.parse_args()

    # ---- ロジックアナライザ：I2C と UART
    cap = la.Capture(a.la, a.fs)
    ops = la.i2c_ops(la.decode_i2c(cap, 0, 1))
    forces = np.array([op[4] for op in ops if op[0] == "write" and op[2] == 0xF4 and op[3] and op[3][0] & 3 == 1])
    stat = [(op[4], op[3][0]) for op in ops if op[0] == "read" and op[2] == 0xF3 and op[3]]
    rd = [(op[4], op[3]) for op in ops if op[0] == "read" and op[2] == 0xF7 and len(op[3]) == 8]
    lines = [(t0, t1, s) for t0, t1, s, ne in la.uart_lines(la.decode_uart(cap, 2), cap.fs) if s.startswith("T,") and ne == 0]
    print(f"LA：{cap.seconds():.1f} 秒、強制モードの書き込み {len(forces)}、状態の読み出し {len(stat)}、"
          f"データの読み出し {len(rd)}、UART の T 行 {len(lines)}")
    rd_t = np.array([t for t, _ in rd])
    st_t = np.array([t for t, _ in stat])
    st_v = np.array([v for _, v in stat])
    L = {}
    for t0, t1, s in lines:
        p = s.split(",")
        if len(p) < 3:
            continue
        seq, adc_t_uart = int(p[1]), int(p[2])
        k = np.searchsorted(rd_t, t0) - 1
        if k < 0 or (t0 - rd_t[k]) / cap.fs > 0.05:
            continue
        d = rd[k][1]
        adc_p = (d[0] << 12) | (d[1] << 4) | (d[2] >> 4)
        adc_t = (d[3] << 12) | (d[4] << 4) | (d[5] >> 4)
        adc_h = (d[6] << 8) | d[7]
        kf = np.searchsorted(forces, rd_t[k]) - 1
        if kf < 0:
            continue
        tf = forces[kf]
        m = (st_t > tf) & (st_t < rd_t[k])
        busy = st_t[m][(st_v[m] & 0x08) != 0]
        ready = st_t[m][(st_v[m] & 0x08) == 0]
        L[seq] = dict(force=tf / cap.fs, rd=rd_t[k] / cap.fs, uart_end=t1 / cap.fs,
                      conv_lo=((busy.max() if len(busy) else tf) - tf) / cap.fs,
                      conv_hi=((ready.min() if len(ready) else rd_t[k]) - tf) / cap.fs,
                      adc=(adc_t, adc_p, adc_h), uart_ok=adc_t == adc_t_uart)
    print(f"  UART の行と I2C の読み出しを組にできたサンプル {len(L)}（UART の adc_t と I2C のバイトが一致 "
          f"{sum(v['uart_ok'] for v in L.values())}）")

    # ---- Pi の記録
    pi_u = {}
    with open(a.uart, newline="") as f:
        for row in csv.DictReader(f):
            p = row["line"].split(",")
            if p[0] == "T" and len(p) >= 2 and p[1].isdigit():
                pi_u[int(p[1])] = float(row["pi_unix"])
    pc, dup = pcap_by_seq(a.pcap)
    db = pd.read_csv(a.db)
    db = db[db["dev"] == DEV].drop_duplicates("seq", keep="first").set_index("seq")
    print(f"Pi：UART の受信 {len(pi_u)} 行、pcap の PUBLISH {len(pc)} 通（同じ seq の2回目以降 {dup}）、DB {len(db)} 行")

    seqs = sorted(s for s in L if s in pi_u and s in pc and s in db.index)
    if len(seqs) < 5:
        sys.exit("4つの記録に共通するサンプルが少なすぎます（記録の時間が重なっているか、dev の名前を確認）")
    # ---- LA → Pi の時計
    la_end = np.array([L[s]["uart_end"] for s in seqs])
    pi_end = np.array([pi_u[s] for s in seqs])
    to_pi, b1, r1 = lower_fit(la_end, pi_end)
    print(f"\n[LA → Pi の時計] 傾き − 1 = {(b1 - 1) * 1e6:+.1f} ppm（2つの水晶の進みの差）、"
          f"UART の受信の遅れ（最小を0とした）中央値 {q(r1, 50) * 1e3:.3f} ms・95%点 {q(r1, 95) * 1e3:.3f} ms")
    # ---- Pico の ticks → LA
    pay = {s: pc[s][1] for s in seqs}
    tk_rd = unwrap_ticks([pay[s]["tk_rd"] for s in seqs])
    tk_pub = tk_rd + (np.array([pay[s]["tk_pub"] - pay[s]["tk_rd"] for s in seqs]) % TICKS_PERIOD)
    la_rd = np.array([L[s]["rd"] for s in seqs])
    to_la, b2, r2 = lower_fit(tk_rd * 1e-6, la_rd)
    print(f"[Pico → LA の時計] 傾き − 1 = {(b2 - 1) * 1e6:+.1f} ppm、ticks_us を呼んでから START まで（最小を0とした）"
          f"中央値 {q(r2, 50) * 1e6:.1f} µs・95%点 {q(r2, 95) * 1e6:.1f} µs")

    # ---- 値の照合
    ok_raw = ok_db = ok_tc = 0
    comp = None
    if a.calib:
        sys.path.insert(0, str(pathlib.Path(__file__).resolve().parents[2] / "lib" / "pico"))
        from bme280 import BME280
        comp = BME280.__new__(BME280)
        comp.has_humidity = False
        for k, v in json.load(open(a.calib)).items():
            setattr(comp, k, v)
    for s in seqs:
        p, d = pay[s], json.loads(db.loc[s, "payload"])
        ok_raw += (p["adc_t"], p["adc_p"], p["adc_h"]) == L[s]["adc"]
        ok_db += all(d.get(k) == p.get(k) for k in ("adc_t", "adc_p", "temp_c", "press_pa"))
        if comp is not None:
            t_c, _, _ = comp.compensate(L[s]["adc"][0], L[s]["adc"][1], None)
            ok_tc += abs(t_c - p["temp_c"]) < 0.005     # Pico の float は単精度のことがある
    print(f"\n[値の照合] {len(seqs)} サンプル：I2C のバイト = 送った JSON の adc {ok_raw}、送った JSON = DB の JSON {ok_db}"
          + (f"、I2C のバイトから計算し直した温度 = Pico の temp_c {ok_tc}" if comp is not None else ""))

    # ---- 各層の時刻（すべて Pi の時計に直す）
    T = pd.DataFrame(index=seqs)
    T["force"] = to_pi([L[s]["force"] for s in seqs])
    T["rd"] = to_pi(la_rd)
    T["uart_end"] = to_pi(la_end)
    T["pub"] = to_pi(to_la(tk_pub * 1e-6))
    T["pcap"] = [pc[s][0] for s in seqs]
    T["db"] = [float(db.loc[s, "recv_unix"]) for s in seqs]
    lay = pd.DataFrame(index=seqs)
    lay["変換（強制モード〜読み出し）"] = T["rd"] - T["force"]
    lay["I2C の読み出し〜UART の行の終わり"] = T["uart_end"] - T["rd"]
    lay["publish〜Pi の eth0（Wi-Fi・中継機）"] = T["pcap"] - T["pub"]
    lay["eth0〜DB（Mosquitto・mqtt2pg）"] = T["db"] - T["pcap"]
    lay["合計：強制モード〜DB"] = T["db"] - T["force"]
    conv = np.array([[L[s]["conv_lo"], L[s]["conv_hi"]] for s in seqs])
    print("\n[各層の遅れ] 中央値（5%点〜95%点）[ms]")
    for c in lay.columns:
        x = lay[c].values * 1e3
        print(f"  {c:30s} {np.median(x):9.3f}（{q(x, 5):.3f}〜{q(x, 95):.3f}）")
    print(f"  変換の終わり（BME280 の measuring ビットが 0 になった時刻）は、強制モードから "
          f"{np.median(conv[:, 0]) * 1e3:.2f}〜{np.median(conv[:, 1]) * 1e3:.2f} ms の間（状態を読む間隔ぶんしか決まらない）")
    print(f"  DB〜Grafana：自動更新 {a.refresh:g} 秒なら、画面に出るまで 0〜{a.refresh:g} 秒"
          f"（更新の時刻と無関係なら一様分布で平均 {a.refresh / 2:g} 秒）＋ 問い合わせの時間。この章の記録からは測れない")

    # ---- 1サンプルを詳しく
    e2e = (T["db"] - T["pub"]).values
    s0 = a.seq if a.seq in T.index else seqs[int(np.argsort(e2e)[len(e2e) // 2])]
    r = T.loc[s0]
    p = pay[s0]
    print(f"\n[サンプル seq={s0} の一生]（Pi の時計、強制モードの書き込みを 0 とした ms）")
    for name, key in [("強制モードの書き込み（I2C、LA）", "force"), ("データの読み出しの START（I2C、LA）", "rd"),
                      ("UART の行の終わり（LA）", "uart_end"), ("publish の直前（Pico の ticks_us）", "pub"),
                      ("Pi の eth0 に PUBLISH が届く（tcpdump）", "pcap"), ("DB に入る（mqtt2pg の recv_unix）", "db")]:
        print(f"  {name:38s} {(r[key] - r['force']) * 1e3:10.3f}")
    print(f"  値：adc_t={p['adc_t']} adc_p={p['adc_p']} → temp_c={p['temp_c']} °C、press_pa={p['press_pa']:.2f} Pa")
    print(f"  Grafana：{a.refresh:g} 秒ごとの更新なら、DB に入ってから 0〜{a.refresh:g} 秒後に画面に出る")

    # ---- 誤差の見積もり
    print("\n[時刻の対応づけの誤差]")
    print(f"  LA の量子化：1/fs = {1e6 / cap.fs:.2f} µs")
    print(f"  LA → Pi：受信の遅れの最小値（未知。短い行は受信タイムアウトの割り込み（約32ビット時間 ≈ 0.28ms）で受け取るのがふつうなので、0.3ms 程度かそれ以上。要確認）だけ LA の出来事が遅めに出る。"
          f"当てはめの下側 20% のばらつき {np.std(np.sort(r1)[:max(3, len(r1) // 5)]) * 1e6:.0f} µs")
    print(f"  Pico → LA：ticks_us から START までの遅れの最小値（未知、数十µs）。当てはめの残差の 95%点 {q(r2, 95) * 1e6:.0f} µs")
    print("  → ms の単位の遅れ（Wi-Fi、DB）には十分小さい。µs の単位の比較（I2C の中のタイミング）は LA の中だけで行う")

    # ---- 統計
    x = lay["publish〜Pi の eth0（Wi-Fi・中継機）"].values
    rho = np.corrcoef(x[:-1], x[1:])[0, 1]
    print(f"\n[統計] Wi-Fi の遅れの 1つずれの自己相関 ρ1 = {rho:.2f}（1秒おきのサンプル）")
    parts = ["publish〜Pi の eth0（Wi-Fi・中継機）", "eth0〜DB（Mosquitto・mqtt2pg）"]
    som = sum(np.median(lay[c]) for c in parts) * 1e3
    mos = np.median(lay[parts].sum(axis=1)) * 1e3
    s95 = sum(q(lay[c], 95) for c in parts) * 1e3
    m95 = q(lay[parts].sum(axis=1), 95) * 1e3
    print(f"  【誤用の例】各層の中央値の和 {som:.3f} ms と、和の中央値 {mos:.3f} ms は一致しない。"
          f"95%点では {s95:.3f} ms と {m95:.3f} ms。")
    print("    中央値や分位点は足し算できない（足せるのは平均だけ）。全体の遅れは、サンプルごとに足してから分位点をとる")
    print("  【誤用の例】Pico の ticks_us と Pi の recv_unix をそのまま引くのは無意味（起点も進み方も違う時計）")

    # ---- 図
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.2, 1]})
    names = ["force", "rd", "uart_end", "pub", "pcap", "db"]
    lab = ["強制モード", "読み出し", "UART の終わり", "publish", "eth0（tcpdump）", "DB"]
    xs = [(r[k] - r["force"]) * 1e3 for k in names]
    ax[0].plot(xs, range(len(xs)), "o-", color="#4a6a90")
    for i, (xx, ll) in enumerate(zip(xs, lab)):
        ax[0].text(xx, i + 0.15, f" {ll} {xx:.2f}", fontsize=8)
    ax[0].set_yticks([])
    ax[0].set_xlabel("強制モードの書き込みからの時間 [ms]（Pi の時計）")
    ax[0].set_title(f"seq={s0} の1サンプル")
    for i, c in enumerate(lay.columns[:4]):
        x = lay[c].values * 1e3
        ax[1].plot([q(x, 5), q(x, 95)], [i, i], color="#4a6a90", lw=1.5)
        ax[1].plot([q(x, 25), q(x, 75)], [i, i], color="#4a6a90", lw=6, alpha=0.5)
        ax[1].plot(np.median(x), i, "o", color="#c0504d")
    ax[1].set_yticks(range(4), ["変換", "I2C〜UART", "Wi-Fi", "DB"])
    ax[1].set_xscale("log")
    ax[1].set_xlabel("遅れ [ms]（対数。細線 5%〜95%、太線 25%〜75%、点は中央値）")
    fig.tight_layout()
    fig.savefig("trace.png", dpi=120)
    print("（図：trace.png）")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile home-lab/partII/ii5/synth_capture.py
#!/usr/bin/env python3
"""II-5：実機の前に解析を試すための「合成の記録」を作る（sigrok-cli の binary と同じ形）

置き場所: ~/home-lab/partII/ii5/synth_capture.py
使い方: cd ~/home-lab/partII/ii5 && uv run python synth_capture.py --out synth
  synth/ の中に、実機の手順で作るのと同じ名前のファイルを作る:
    bus.bin（4MHz、D0=SDA D1=SCL D2=Pico TX D3=Pi TX D4=SCK D5=MOSI/MISO D6=CS D7=PWM）、bus_log.csv、bus_i2c.txt
    rise_a.bin・rise_b.bin・rise_c.bin（24MHz、プルアップ抵抗だけを変えた I2C）
    enc.bin（2MHz、D0=A相 D1=B相、手で回したときの速さのむら）
    trace.bin（2MHz、D0=SDA D1=SCL D2=Pico TX）、uart_trace.csv、mqtt_ii5.pcap、db_ii5.csv
  最後に「正解」（作るときに使った値）を表示するので、解析の結果と比べる。

しくみ：線ごとに「ロジックアナライザのしきい値を横切る時刻」（本当の時刻、秒）を作り、
ロジックアナライザのサンプル時刻（水晶のずれ e_la とランダムな位相つき）で値を読む。
I2C の立ち上がりは RC の充電 V = VDD(1 − e^(−t/RC)) がしきい値に届く時刻、
I2C のクロックの High の時間は「Pico が High を感じてから数え始める」とした（II-5-3-2）。
"""
import argparse
import csv
import json
import pathlib
import struct
import sys

import numpy as np

sys.path.insert(0, str(pathlib.Path(__file__).resolve().parents[2] / "lib" / "pico"))
from bme280 import BME280  # noqa: E402  入門3 の補正式をそのまま使う

rng = np.random.default_rng(5)
VDD = 3.3
E_PICO, E_LA = 12e-6, -18e-6          # 水晶のずれ（Pico、ロジックアナライザ）
FCLK = 150e6 * (1 + E_PICO)           # Pico のクロック（本当の Hz）
CAL = dict(T1=27504, T2=26435, T3=-1000, P1=36477, P2=-10685, P3=3024, P4=2855,
           P5=140, P6=-7, P7=15500, P8=-14600, P9=6000)       # 入門3 の test_bme280_vector.py と同じ値
CAL_BYTES = struct.pack("<HhhHhhhhhhhh", *CAL.values())


class Line:
    """1本の線の「しきい値を横切る時刻と、そのあとの値」"""

    def __init__(self, init=1):
        self.v, self.t, self.x = init, [], []

    def set(self, t, v):
        if v != self.v:
            self.t.append(t)
            self.x.append(v)
            self.v = v


def sample(lines, fs, seconds):
    """ロジックアナライザで読む。lines は {ビット番号: Line}"""
    n = int(fs * seconds)
    phase = rng.random()
    out = np.zeros(n, dtype=np.uint8)
    ts = (np.arange(n) + phase) / (fs * (1 + E_LA))
    for bit, ln in lines.items():
        order = np.argsort(ln.t, kind="stable")
        t = np.asarray(ln.t)[order]
        x = np.asarray(ln.x, dtype=np.uint8)[order]
        init = ln.x[0] ^ 1 if ln.x else ln.v
        k = np.searchsorted(t, ts, side="right")
        val = np.where(k == 0, init, x[np.maximum(k - 1, 0)])
        out |= (val.astype(np.uint8) << bit)
    return out


class I2CBus:
    """DW_apb_i2c に似せた I2C の親（Pico）。freq は 100k か 400k"""

    def __init__(self, sda, scl, rp, cb, freq=100_000, x_la=0.45, x_pico=0.55):
        self.sda, self.scl = sda, scl
        period = int((150e6 + freq // 2) // freq)              # pico-sdk の i2c_set_baudrate と同じ考え方
        lcnt = period * 3 // 5
        hcnt = period - lcnt
        spk = 1 if lcnt < 16 else lcnt // 16
        self.t_low = (lcnt + 1) / FCLK
        self.t_high = (hcnt + spk + 7) / FCLK
        rc = rp * cb
        self.d_la = rc * np.log(1 / (1 - x_la))               # 手を離してから LA のしきい値まで
        self.d_pico = rc * np.log(1 / (1 - x_pico))           # 手を離してから Pico が High を感じるまで
        self.t = 0.0

    def _sda(self, t, v):
        self.sda.set(t + (self.d_la if v else 5e-9), v)

    def _clock(self, bit_from_slave=False, v=1):
        """SCL が Low の間に SDA を v にし、SCL を上げて下げる（1ビット）"""
        hold = 0.5e-6 if bit_from_slave else 0.3e-6
        self._sda(self.t + hold, v)
        rel = self.t + self.t_low
        self.scl.set(rel + self.d_la, 1)
        fall = rel + self.d_pico + 2 / FCLK + self.t_high
        self.scl.set(fall + 5e-9, 0)
        self.t = fall

    def start(self, t):
        self.t = max(self.t, t)
        self._sda(self.t, 0)
        self.t += 4.7e-6
        self.scl.set(self.t, 0)

    def rstart(self):
        self._sda(self.t + 0.3e-6, 1)
        rel = self.t + self.t_low
        self.scl.set(rel + self.d_la, 1)
        t_hi = rel + self.d_pico + 4.7e-6
        self._sda(t_hi, 0)
        self.t = t_hi + 4.0e-6
        self.scl.set(self.t, 0)

    def stop(self):
        self._sda(self.t + 0.3e-6, 0)
        rel = self.t + self.t_low
        self.scl.set(rel + self.d_la, 1)
        self._sda(rel + self.d_pico + 4.0e-6, 1)
        self.t = rel + 10e-6

    def byte(self, b, from_slave=False, ack=True):
        for k in range(7, -1, -1):
            self._clock(from_slave, (b >> k) & 1)
        self._clock(not from_slave, 0 if ack else 1)

    def read_mem(self, t, addr, reg, data):
        self.start(t)
        self.byte(addr << 1)
        self.byte(reg)
        self.rstart()
        self.byte((addr << 1) | 1)
        for i, d in enumerate(data):
            self.byte(d, from_slave=True, ack=i < len(data) - 1)
        self.stop()
        return self.t

    def write_mem(self, t, addr, reg, data):
        self.start(t)
        self.byte(addr << 1)
        self.byte(reg)
        for d in data:
            self.byte(d)
        self.stop()
        return self.t


def uart_send(line, t, text, baud, jitter=2e-9):
    T = 1 / baud
    for ch in text.encode():
        bits = [0] + [(ch >> k) & 1 for k in range(8)] + [1]
        for k, b in enumerate(bits):
            line.set(t + k * T + rng.normal(0, jitter), b)
        t += 10 * T + 1e-6 * rng.random()                    # バイトの間のすき間（少し）
    return t


def spi_xfer(sck, mosi, cs, t, data, baud):
    half = 0.5 / baud
    cs.set(t, 0)
    t += half
    for b in data:
        for k in range(7, -1, -1):
            mosi.set(t - half + 20e-9, (b >> k) & 1)
            sck.set(t, 1)
            sck.set(t + half, 0)
            t += 2 * half
    cs.set(t, 1)
    return t + half


def make_bus(out, seconds, fs=4_000_000):
    L = {k: Line(1) for k in range(8)}
    L[4].v = 0
    L[6].v = 1
    L[7].v = 0
    bus = I2CBus(L[0], L[1], rp=4.7e3, cb=80e-12)
    pico_baud = FCLK / (16 * (81 + 24 / 64))                  # 115200 を頼んだときの実際の値
    pi_baud = 115200 * (1 - 86e-6)                            # Pi 側（作った値）
    log, ann, seq, t = [], [], 0, 2e-3
    glitch_seq = 7                                            # わざと1バイトを化けさせる（照合の試験）
    while t < seconds - 0.25:
        seq += 1
        t0 = t
        chip = 0x60
        bus.read_mem(t, 0x76, 0xD0, [chip])
        cal = bytearray(CAL_BYTES)
        if seq == glitch_seq:
            cal[5] ^= 0x10
        bus.read_mem(bus.t + 30e-6, 0x76, 0x88, list(cal))
        bus.read_mem(bus.t + 30e-6, 0x40, 0x00, [0x19, 0x9F])
        bus.read_mem(bus.t + 30e-6, 0x40, 0x05, [0x10, 0x00])
        for a, r, d in [(0x76, 0xD0, [chip]), (0x76, 0x88, list(cal)), (0x40, 0x00, [0x19, 0x9F]), (0x40, 0x05, [0x10, 0x00])]:
            ann.append("i2c-1: Address write: %02X" % a)
            ann.append("i2c-1: Data write: %02X" % r)
            ann.append("i2c-1: Address read: %02X" % a)
            ann += ["i2c-1: Data read: %02X" % x for x in d]
        tu = uart_send(L[2], bus.t + 80e-6, "U,%d,UUUU\n" % seq, pico_baud)
        uart_send(L[3], tu + 0.4e-3 + 2e-3 * rng.random(), "A,%d\n" % seq, pi_baud)
        tx = [0x55, 0xAA, 0x00, 0xFF, 0x0F, 0xF0, seq & 0xFF, 0xA5]
        spi_xfer(L[4], L[5], L[6], tu + 0.2e-3, tx, 500_000 * (1 + E_PICO))
        log.append([f"{1.7e9 + t0:.3f}", seq, int(t0 * 1e6) & 0x3FFFFFFF, 100000, chip,
                    CAL_BYTES.hex(), 0x199F, 0x1000, 0, seq - 1, 0])
        t = t0 + 0.2 / (1 + E_PICO)
    # PWM 1kHz・25%（D7）
    P = 1e-3 / (1 + E_PICO)
    k = np.arange(int(seconds / P))
    for tk in k * P + 0.37e-3:
        L[7].set(tk, 1)
        L[7].set(tk + 0.25 * P, 0)
    sample(L, fs, seconds).tofile(out / "bus.bin")
    with open(out / "bus_log.csv", "w") as f:
        f.write("host_time,seq,tk_us,i2c_hz,chip_id,calib_hex,ina_cfg,ina_cal,spi_biterr,uart_acks,i2c_err\n")
        for r in log:
            f.write(",".join(str(x) for x in r) + "\n")
    (out / "bus_i2c.txt").write_text("\n".join(ann) + "\n")
    return dict(pico_baud=pico_baud, pi_baud=pi_baud, scl_period_us=1e6 * (bus.t_low + bus.t_high + bus.d_pico + 2 / FCLK),
                glitch_seq=glitch_seq, cycles=seq)


def make_rise(out, fs=24_000_000, seconds=1.0, cb=90e-12):
    truth = {}
    for name, rp in [("a", 9.9e3), ("b", 4.95e3), ("c", 2.6e3)]:
        L = {0: Line(1), 1: Line(1)}
        bus = I2CBus(L[0], L[1], rp=rp, cb=cb)
        t = 1e-3
        while t < seconds - 5e-3:
            bus.read_mem(t, 0x76, 0x88, list(CAL_BYTES))
            t = bus.t + 1e-3
        sample(L, fs, seconds).tofile(out / f"rise_{name}.bin")
        truth[name] = dict(rp=rp, low_us=1e6 * (bus.t_low + bus.d_la), tr_30_70_ns=1e9 * 0.8473 * rp * cb)
    return truth


def make_enc(out, fs=2_000_000, seconds=10.0, ppr=7, phase_deg=80.0, duty_a=0.48, duty_b=0.53):
    dt = 1e-6
    n = int(seconds / dt)
    # 手で回す：回転数 [回転/s] がなめらかにゆらぐ（AR(1) をさらに平滑化）
    w = np.zeros(n // 1000)
    for i in range(1, len(w)):
        w[i] = 0.995 * w[i - 1] + rng.normal(0, 0.4)
    w = 25 + 15 * np.tanh(w / 4)                         # モータ軸で 10〜40 回転/s
    w = np.interp(np.arange(n) / 1000, np.arange(len(w)), w)
    theta = np.cumsum(w) * dt * ppr                        # 単位：エンコーダの周期
    L = {0: Line(0), 1: Line(0)}
    for line, ofs, duty in [(L[0], 0.0, duty_a), (L[1], phase_deg / 360, duty_b)]:
        ph = theta - ofs
        st = ((ph - np.floor(ph)) < duty).astype(np.int8)
        k = np.flatnonzero(np.diff(st)) + 1
        line.v = int(st[0])
        for kk in k:
            line.t.append(kk * dt + rng.normal(0, 1.5e-6))
            line.x.append(int(st[kk]))
    sample(L, fs, seconds).tofile(out / "enc.bin")
    return dict(phase_deg=phase_deg, duty_a=duty_a, duty_b=duty_b)


def pcap_write(path, pkts):
    """pkts: [(Pi の UNIX 時刻, TCP の中身)]。Ethernet + IPv4 + TCP の最小限の形"""
    with open(path, "wb") as f:
        f.write(struct.pack("<IHHiIII", 0xA1B2C3D4, 2, 4, 0, 0, 65535, 1))
        seqn = 1000
        for t, pl in pkts:
            tcp = struct.pack(">HHIIBBHHH", 50123, 1883, seqn, 1, 5 << 4, 0x18, 5840, 0, 0) + pl
            ip = struct.pack(">BBHHHBBH4s4s", 0x45, 0, 20 + len(tcp), 1, 0, 64, 6, 0,
                             bytes([192, 168, 1, 60]), bytes([192, 168, 1, 50])) + tcp
            eth = b"\x02" * 6 + b"\x04" * 6 + b"\x08\x00" + ip
            sec = int(t)
            f.write(struct.pack("<IIII", sec, int(round((t - sec) * 1e6)), len(eth), len(eth)) + eth)
            seqn += len(pl)


def mqtt_publish(topic, payload):
    body = struct.pack(">H", len(topic)) + topic + payload
    n, rem = len(body), b""
    while True:
        d = n % 128
        n //= 128
        rem += bytes([d | (0x80 if n else 0)])
        if not n:
            break
    return b"\x30" + rem + body


def make_trace(out, fs=2_000_000, seconds=60.0):
    L = {0: Line(1), 1: Line(1), 2: Line(1)}
    bus = I2CBus(L[0], L[1], rp=4.7e3, cb=80e-12)
    pico_baud = FCLK / (16 * (81 + 24 / 64))
    comp = BME280.__new__(BME280)                      # I2C なしで補正式だけを使う
    comp.has_humidity = False
    for k, v in CAL.items():
        setattr(comp, k, v)
    PI0, E_PI = 1.7634e9 + 123.456, 5e-6              # Pi の時計 = PI0 + (1+E_PI)·本当の時刻
    TK0 = 987654321                                    # Pico の ticks_us の起点（適当）
    pkts, rows, uart_rows, truth = [], [], [], []
    t, seq, rid = 0.5, 0, 50000

    def pi_clock(x):
        return PI0 + (1 + E_PI) * x
    while t < seconds - 1.0:
        seq += 1
        t_force = t
        bus.write_mem(t_force, 0x76, 0xF4, [0x25])
        tt = bus.t + 2e-3
        conv_end = t_force + 9.3e-3 * (0.95 + 0.1 * rng.random())
        while True:
            busy = tt < conv_end
            bus.read_mem(tt, 0x76, 0xF3, [0x08 if busy else 0x00])
            if not busy:
                break
            tt = bus.t + 1.0e-3
        adc_t = 519888 + int(rng.normal(0, 30)) + seq * 3
        adc_p = 415148 + int(rng.normal(0, 50))
        adc_h = 27000 + int(rng.normal(0, 20))
        d = [adc_p >> 12 & 0xFF, adc_p >> 4 & 0xFF, (adc_p & 0xF) << 4, adc_t >> 12 & 0xFF, adc_t >> 4 & 0xFF,
             (adc_t & 0xF) << 4, adc_h >> 8, adc_h & 0xFF]
        t_rd = bus.t + 60e-6
        tk_rd = (TK0 + int(round((t_rd - 25e-6 - abs(rng.normal(0, 8e-6))) * 1e6 * (1 + E_PICO)))) & 0x3FFFFFFF
        bus.read_mem(t_rd, 0x76, 0xF7, d)
        t_c, p_pa, _ = comp.compensate(adc_t, adc_p, None)
        tu0 = bus.t + 0.35e-3
        tu1 = uart_send(L[2], tu0, "T,%d,%d\n" % (seq, adc_t), pico_baud)
        t_pub = tu0 + 0.15e-3
        tk_pub = (TK0 + int(round(t_pub * 1e6 * (1 + E_PICO)))) & 0x3FFFFFFF
        lat_uart = 0.30e-3 + rng.exponential(0.5e-3)      # 受信タイムアウトの割り込み（約32ビット時間）＋ Python
        uart_rows.append((pi_clock(tu1 + lat_uart), "T,%d,%d" % (seq, adc_t)))
        wifi = 3e-3 + rng.lognormal(np.log(4e-3), 0.5) + (rng.random() < 0.05) * rng.exponential(40e-3)
        t_pcap = t_pub + 1.5e-3 + wifi
        msg = {"dev": "pico1-ii5", "seq": seq, "adc_t": adc_t, "adc_p": adc_p, "adc_h": adc_h,
               "temp_c": t_c, "press_pa": p_pa, "hum_pct": 48.2, "tk_rd": tk_rd, "tk_pub": tk_pub, "nf": 0}
        pl = json.dumps(msg).encode()
        pkts.append((pi_clock(t_pcap), mqtt_publish(b"lab/pico1-ii5/telemetry", pl)))
        if seq % 10 == 0:
            pkts.append((pi_clock(t_pcap + 0.3), b"\xc0\x00"))         # PINGREQ（関係のないパケット）
        if seq == 17:                                                   # 再送（同じ中身がもう一度）
            pkts.append((pi_clock(t_pcap + 0.2), mqtt_publish(b"lab/pico1-ii5/telemetry", pl)))
        t_db = t_pcap + 0.6e-3 + rng.exponential(0.8e-3)
        rid += 1
        rows.append((rid, pi_clock(t_db), "lab/pico1-ii5/telemetry", "pico1-ii5", seq, json.dumps(msg)))
        truth.append(dict(seq=seq, wifi_ms=1e3 * (t_pcap - t_pub), db_ms=1e3 * (t_db - t_pcap),
                          conv_ms=1e3 * (t_rd - t_force), uart_lat_ms=1e3 * lat_uart))
        t += 1.0 / (1 + E_PICO)
    sample(L, fs, seconds).tofile(out / "trace.bin")
    pkts.sort()
    pcap_write(out / "mqtt_ii5.pcap", pkts)
    with open(out / "uart_trace.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["pi_unix", "line"])
        for tp, ln in uart_rows:
            w.writerow([f"{tp:.6f}", ln])
    with open(out / "db_ii5.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id", "recv_unix", "topic", "dev", "seq", "payload"])
        for r in rows:
            w.writerow([r[0], f"{r[1]:.6f}", r[2], r[3], r[4], r[5]])
    wifi = np.array([x["wifi_ms"] for x in truth])
    dbm = np.array([x["db_ms"] for x in truth])
    um = np.array([x["uart_lat_ms"] for x in truth])
    return dict(n=seq, wifi_median_ms=float(np.median(wifi)), db_median_ms=float(np.median(dbm)),
                uart_lat_min_ms=float(um.min()), slope_pi_vs_la_ppm=1e6 * ((1 + E_PI) / (1 + E_LA) - 1),
                slope_la_vs_pico_ppm=1e6 * ((1 + E_LA) / (1 + E_PICO) - 1))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="synth")
    ap.add_argument("--bus-seconds", type=float, default=4.0)
    a = ap.parse_args()
    out = pathlib.Path(a.out)
    out.mkdir(exist_ok=True)
    print("bus:", make_bus(out, a.bus_seconds))
    print("rise:", make_rise(out))
    print("enc:", make_enc(out))
    print("trace:", make_trace(out))
    print("（E_PICO = %+.0f ppm、E_LA = %+.0f ppm）" % (E_PICO * 1e6, E_LA * 1e6))


if __name__ == "__main__":
    main()

In [ ]:
import glob, os, re, shlex, subprocess, sys
from IPython.display import Image, display

os.chdir(os.path.join(os.environ["II5_NB_HOME"], "home-lab/partII/ii5"))   # 教科書の cd ~/home-lab/partII/ii5 と同じ


def run(args):
    """ターミナルの「uv run python ○○.py …」の代わり。表示を出し、あとで確かめられるように文字列で返す"""
    words = []
    for w in shlex.split(args):
        words += (sorted(glob.glob(w)) or [w]) if any(c in w for c in "*?[") else [w]
    p = subprocess.run([sys.executable] + words, capture_output=True, text=True)
    print(p.stdout, end="")
    err = "\n".join(l for l in p.stderr.splitlines() if "findfont" not in l and "font cache" not in l)
    if err.strip():
        print(err, file=sys.stderr)
    assert p.returncode == 0, "★実行に失敗：" + args
    return p.stdout


def show(*paths, width=800):
    """保存された図を表示する"""
    for f in paths:
        print(f)
        display(Image(filename=f, width=width))


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def num(pattern, text, i=1):
    """表示（text）から、pattern の (…) の数を取り出す"""
    m = re.search(pattern, text)
    assert m, "表示に見つからない：" + pattern
    return float(m.group(i).replace("−", "-"))


print("準備ができました。作業フォルダ:", os.getcwd())

## 1. 合成の記録を作る（手順 A）

`synth_capture.py --out synth` が、実機の手順で作るのと同じ名前のファイルを `synth/` に作ります（数秒〜十数秒）。最後の数行が「正解」（作るときに使った値）です。以下の節では、解析の結果をこの正解と比べます。

| ファイル | 中身 |
|---|---|
| `bus.bin` | 4 MHz・4 秒。D0 = SDA、D1 = SCL、D2 = Pico の TX、D3 = Pi の TX、D4 = SCK、D5 = MOSI/MISO、D6 = CS、D7 = PWM |
| `bus_log.csv`・`bus_i2c.txt` | Pico が自分で読んだ値の記録、sigrok-cli の I2C デコーダの注釈と同じ形のもの |
| `rise_a.bin`・`rise_b.bin`・`rise_c.bin` | 24 MHz。プルアップ抵抗だけを変えた I2C（9.9 kΩ・4.95 kΩ・2.6 kΩ） |
| `enc.bin` | 2 MHz・10 秒。エンコーダの A 相・B 相（手で回したような速さのむら） |
| `trace.bin`・`uart_trace.csv`・`mqtt_ii5.pcap`・`db_ii5.csv` | 最終節の1サンプルを追う記録（60 秒） |

In [ ]:
out = run("synth_capture.py --out synth")
truth = {k: v for k, v in re.findall(r"^(\w+): (\{.*\})$", out, re.M)}
for k in ("bus", "rise", "enc", "trace"):
    ok(k in truth, f"正解の行 {k} が表示された")
T_SCL = num(r"'scl_period_us': (?:np\.float64\()?([\d.]+)", truth["bus"])
T_WIFI = num(r"'wifi_median_ms': (?:np\.float64\()?([\d.]+)", truth["trace"])
T_UARTMIN = num(r"'uart_lat_min_ms': (?:np\.float64\()?([\d.]+)", truth["trace"])
T_PHASE = num(r"'phase_deg': ([\d.]+)", truth["enc"])
T_DUTY = num(r"'duty_a': ([\d.]+)", truth["enc"])
for f in ["bus.bin", "bus_log.csv", "bus_i2c.txt", "rise_a.bin", "rise_b.bin", "rise_c.bin", "enc.bin",
          "trace.bin", "uart_trace.csv", "mqtt_ii5.pcap", "db_ii5.csv"]:
    assert os.path.exists("synth/" + f), f
ok(True, "synth/ に 11 個のファイルができた")
ok(os.path.getsize("synth/bus.bin") == 4_000_000 * 4, "bus.bin は 4 MHz × 4 秒 = 1600 万サンプル（1サンプル1バイト）")

## 2. 自前のデコーダで読む（`la_decode.py`）

教科書の `la_decode.py` は、sigrok のデコーダを使わずに、記録のバイト列から I2C・UART・SPI・エンコーダを読みます（II-5-3）。まず I2C の最初の4件を見ます。BME280（0x76）のチップID（0xD0）と補正係数（0x88〜）、INA219（0x40）の設定（0x00）とキャリブレーション（0x05）を読んでいるはずです。

In [ ]:
out = run("la_decode.py i2c synth/bus.bin 4000000 --max 4")
ok("SDA と SCL が同じサンプルで変わった箇所 0" in out, "SDA と SCL が同じサンプルで変わった所がない（4 MHz なら十分速い）")
ok(re.search(r"read  0x76 reg=0xD0 \[60\]", out) is not None, "BME280（0x76）の 0xD0 から 0x60 が読めた")
ok(re.search(r"read  0x40 reg=0x00 \[19 9F\]", out) is not None, "INA219（0x40）の設定 0x199F が読めた")
ok(re.search(r"read  0x40 reg=0x05 \[10 00\]", out) is not None, "INA219 のキャリブレーション 0x1000 が読めた")

UART（D2 = Pico の TX）、SPI（D4〜D6）、エンコーダ（`enc.bin`）も同じプログラムで読みます。

In [ ]:
out = run("la_decode.py uart synth/bus.bin 4000000 --ch 2 --max 3")
ok("ストップビットの誤り 0" in out, "UART（D2）のストップビットの誤りが 0")
out = run("la_decode.py spi synth/bus.bin 4000000 --max 3")
ok("送るはずの並び（55 AA 00 FF 0F F0 seq A5）と違う区間 0 個" in out and "seq が1ずつ増えていない所 0 個" in out,
   "SPI の D5 のバイト列が、全区間で送るはずの並びと一致")
out = run("la_decode.py enc synth/enc.bin 2000000")
ok("2段とび 0 回" in out, "エンコーダの2段とび（数えそこない）が 0 回")

## 3. レジスタを照らし合わせる（`check_regs.py`）

手順 A の「確かめること」：`check_regs.py` が「チップID 0x60 → BME280」「0x199F … 一致」「読み出し 72 回のうち食い違い 1 回」と出し、sigrok 形式の注釈との比較が「一致 76 件」。

合成の記録では、補正係数の読み出しの1回だけ、1バイト（0xFC → 0xEC）をわざと化けさせてあります（正解の `glitch_seq`）。食い違いの割合の区間は、**読み出し1回を単位**にした Clopper–Pearson と Wilson です。ビットを単位にすると区間が狭く見える誤用も並べて表示されます（II-5-3-8）。

In [ ]:
out = run("check_regs.py synth/bus.bin 4000000 --log synth/bus_log.csv --ann synth/bus_i2c.txt --calib-out synth/calib.json")
ok("チップID 0x60 → BME280" in out, "チップID 0x60 → BME280")
ok(re.search(r"0x00（設定）= 0x199F .*ina219.py の値と一致", out) is not None, "INA219 の設定 0x199F が ina219.py の値と一致")
ok(re.search(r"0x05（キャリブレーション）= 0x1000 .*一致", out) is not None, "キャリブレーション 0x1000 も一致")
ok("読み出し 72 回のうち食い違い 1 回" in out, "読み出し 72 回のうち食い違い 1 回（本文の手順 A と同じ）")
ok(re.search(r"一致 76 件", out) is not None, "sigrok 形式の注釈との比較が一致 76 件")
glitch = int(num(r"'glitch_seq': (\d+)", truth["bus"]))
ok(f"食い違い：seq={glitch} " in out, f"食い違いは、わざと化けさせた seq={glitch} の読み出し")
lo, hi = num(r"Clopper–Pearson ([\d.]+)〜", out), num(r"Clopper–Pearson [\d.]+〜([\d.]+) %", out)
from scipy import stats
ok(abs(lo - stats.beta.ppf(0.025, 1, 72) * 100) < 0.01 and abs(hi - stats.beta.ppf(0.975, 2, 71) * 100) < 0.01,
   f"Clopper–Pearson の区間 {lo}〜{hi} % を scipy の β 分布で計算し直しても同じ")
bit_hi = num(r"上限 ([\d.]+) %", out)
ok(bit_hi < hi / 10, f"ビットを単位にした上限 {bit_hi} % は、読み出し単位の上限 {hi} % の 1/10 より小さい（狭すぎる＝誤用）")
ok(os.path.exists("synth/calib.json"), "補正係数を synth/calib.json に保存した（7 節の trace_join.py が使う）")

本文 II-5-7 の「600回なら上限は約0.6%（Clopper–Pearson）」「ビットを単位にすると上限が数十分の一（約1/58）に見える」も、式で確かめておきます（食い違い 0 回のとき。1回の読み出しを 58 ビットとみなした比です）。

In [ ]:
from scipy import stats
up600 = stats.beta.ppf(0.975, 1, 600) * 100
ok(abs(up600 - 0.6) < 0.05, f"600 回で食い違い 0 回のときの Clopper–Pearson の上限 = {up600:.3f} %（約 0.6 %）")
up_bit = stats.beta.ppf(0.975, 1, 600 * 58) * 100
print(f"同じ 600 回を 58 ビットずつのビットに数えると、上限 {up_bit:.4f} %（比 {up600 / up_bit:.0f} 分の 1）")
ok(50 < up600 / up_bit < 66, "ビットで数えた上限は、読み出しで数えた上限の約 1/58")

## 4. 周期・ボーレートを区間推定する（`bus_stats.py`）

ロジックアナライザは、しきい値を横切った時刻を1サンプルの幅でしか知りません（量子化）。`bus_stats.py` は、1周期のサンプル数が「隣り合う2つの値だけ」をとることを確かめ、やりとり（あるいはフレーム）を単位にして区間を出します（II-5-3-6）。

II-5-7 の結果の例：「SCL：設定は 100kHz でも、測ると 90kHz 台（合成データでは 93.1kHz）。1周期の値は隣り合う2つのサンプル数だけをとり、やりとりを単位にした区間は 1ns より狭い」。

In [ ]:
out = run("bus_stats.py i2c synth/bus.bin 4000000")
f_scl = num(r"SCL ([\d.]+) kHz", out)
ok(abs(f_scl - 93.1) < 0.05, f"SCL = {f_scl} kHz（本文の合成データの値 93.1 kHz）")
ok(abs(f_scl - 1e3 / T_SCL) < 0.02, f"正解（周期 {T_SCL:.4f} µs → {1e3 / T_SCL:.3f} kHz）と一致")
lo, hi = num(r"周期 = [\d.]+ µs（95% 区間 ([\d.]+)〜", out), num(r"周期 = [\d.]+ µs（95% 区間 [\d.]+〜([\d.]+)", out)
ok((hi - lo) * 1000 < 1.0, f"やりとりを単位にした周期の区間の幅 {(hi - lo) * 1000:.2f} ns は 1 ns より狭い")
vals = re.search(r"1周期 のとる値（サンプル数: 回数）: (.*)", out).group(1)
ks = [int(k) for k in re.findall(r"(\d+): \d+", vals)]
ok(len(ks) == 2 and ks[1] - ks[0] == 1, f"1周期のサンプル数は隣り合う2つの値だけ（{vals}）")
show("stats_i2c.png")

UART の2本（D2 = Pico、D3 = Pi）のボーレートです。II-5-7：「Pico・Pi とも 115200 から数百ppm 以内（ロジックアナライザの水晶のずれを含む）。2つの差は許容範囲 ±4.6% の百分の一以下」。

In [ ]:
out = run("bus_stats.py uart synth/bus.bin 4000000 --ch 2 3")
d2, d3 = num(r"D2：.*ずれ ([-+\d]+) ppm", out), num(r"D3：.*ずれ ([-+\d]+) ppm", out)
ok(abs(d2) < 1000 and abs(d3) < 1000, f"Pico（D2）{d2:+.0f} ppm・Pi（D3）{d3:+.0f} ppm とも 115200 から数百 ppm 以内")
diff = num(r"D2 と D3 の差：([-+\d]+) ppm", out)
ok(abs(diff) * 1e-6 < 0.046 / 100, f"2つの差 {diff:+.0f} ppm は、許容範囲 ±4.6% の百分の一（460 ppm）以下")
show("stats_uart.png")

SPI（SCK）と PWM（D7）です。II-5-7：「SPI：500.000kHz … PWM：1000Hz・25.0%、幅の値はほぼ1つ（4MHz で 1000 サンプル）」。

In [ ]:
out = run("bus_stats.py spi synth/bus.bin 4000000")
ok("SCK = 500.000 kHz" in out, "SPI の SCK = 500.000 kHz")
out = run("bus_stats.py pwm synth/bus.bin 4000000 --ch 7")
f_pwm, duty = num(r"([\d.]+) Hz、量子化", out), num(r"デューティ ([\d.]+) %", out)
ok(abs(f_pwm - 1000) < 0.1 and abs(duty - 25.0) < 0.05, f"PWM = {f_pwm:.3f} Hz・デューティ {duty:.2f} %（1000 Hz・25.0 %）")
ok("幅 のとる値（サンプル数: 回数）: 1000: " in out, "幅は 1000 サンプル（4 MHz で 250 µs）の1つの値")

## 5. プルアップ抵抗と立ち上がり（`rise_compare.py`）

同じ I2C を、プルアップ抵抗だけを変えて 24 MHz で記録したもの（合成では 9.9 kΩ・4.95 kΩ・2.6 kΩ、配線の容量 $C_b = 90$ pF）を比べます。ロジックアナライザには立ち上がりの形は見えませんが、Low の長さと周期が $R$ に比例して伸びる傾き α・β から、容量を**上限と下限**の形で言えます（II-5-3-3）。本文：「合成データ（$C_b = 90$pF）で試すと、$C_b$ は 77〜194pF、$R_p = 9.9$kΩ の $t_r$ は 650〜1630ns 程度」「R を 1kΩ 増やすと Low が数十ns 長くなる（合成データでは 54ns）」。

In [ ]:
out = run("rise_compare.py --fs 24000000 synth/rise_a.bin:9900 synth/rise_b.bin:4950 synth/rise_c.bin:2600")
alpha = num(r"1kΩ 増やすと ([\d.]+) ns 長くなる", out)
ok(abs(alpha - 54) < 1, f"R を 1 kΩ 増やすと Low が {alpha} ns 長くなる（本文の合成データの値 54 ns）")
c_lo, c_hi = num(r"合わせると C = (\d+)〜", out), num(r"合わせると C = \d+〜(\d+) pF", out)
ok(c_lo <= 90 <= c_hi, f"容量の範囲 {c_lo:.0f}〜{c_hi:.0f} pF に正解の 90 pF が入る（本文 77〜194 pF 程度）")
ok(abs(c_lo - 77) <= 3 and abs(c_hi - 194) <= 5, "範囲の両端も本文の値とほぼ同じ")
tr_lo, tr_hi = num(r"R =\s+9900 Ω：\s+(\d+)〜", out), num(r"R =\s+9900 Ω：\s+\d+〜\s*(\d+) ns", out)
ok(abs(tr_lo - 650) < 20 and abs(tr_hi - 1630) < 30, f"9.9 kΩ の t_r = {tr_lo:.0f}〜{tr_hi:.0f} ns（本文 650〜1630 ns 程度）")
ok(re.search(r"9900 Ω：.*400kHz：規格外と言える", out) is not None and re.search(r"9900 Ω：.*100kHz：どちらとも言えない", out) is not None,
   "9.9 kΩ では「400kHz の 300ns は超えている」「100kHz の 1000ns を超えているかは言えない」")
tr_true = num(r"'a': \{'rp': 9900\.0, 'low_us': (?:np\.float64\()?[\d.]+\)?, 'tr_30_70_ns': ([\d.]+)", truth["rise"])
ok(tr_lo <= tr_true <= tr_hi, f"正解の t_r（{tr_true:.0f} ns）も区間に入る")
show("rise_compare.png")

## 6. エンコーダ：位相差・デューティと、続けて測った値（`bus_stats.py enc`）

II-5-7：「エンコーダ：位相差 90°±十数°、デューティ 50%±数%（合成データでは 80°・48%）。手で回した周期の ρ1 は 0.9 以上で、有効サンプル数は数十」。手で回す速さはゆっくり変わるので、A 相の周期は隣どうしがよく似ています（自己相関 ρ1 が 1 に近い）。独立とみなした区間が狭すぎる例です。

In [ ]:
out = run("bus_stats.py enc synth/enc.bin 2000000")
ph = num(r"位相差 \[°\]: 平均 ([\d.]+)", out)
du = num(r"A 相のデューティ: 平均 ([\d.]+)", out)
ok(abs(ph - T_PHASE) < 0.5 and abs(du - T_DUTY) < 0.005, f"位相差 {ph:.1f}°・デューティ {du * 100:.1f} %（正解 {T_PHASE:.0f}°・{T_DUTY * 100:.0f} %）")
blk = out[out.index("A 相の周期 [ms]"):]
rho, neff = num(r"ρ1 = ([\d.]+)", blk), num(r"有効サンプル数 n\(1−ρ\)/\(1\+ρ\) ≈ (\d+)", blk)
ok(rho >= 0.9 and 10 <= neff < 100, f"A 相の周期の ρ1 = {rho:.2f}（0.9 以上）、有効サンプル数 ≈ {neff:.0f}（数十）")
show("stats_enc.png")

## 7. 1サンプルを各層で追う（`trace_join.py`）

ロジックアナライザ（I2C と UART）、Pi の UART の受信時刻、`tcpdump` の pcap、DB の行を、3つの時計（LA・Pico・Pi）の対応づけでつなぎ、BME280 の1回の測定が DB に入るまでの各層の遅れを出します（II-5-3-7）。

手順 A の「確かめること」：「trace_join.py の Wi-Fi の遅れの中央値が、正解（8.4ms 前後）より 0.3ms ほど短い（II-5-3-7 の片側の誤差。合成データでは受信の遅れの最小値を 0.3ms にした）」。LA の出来事を Pi の時計に直すとき、UART の受信の遅れの最小を 0 とみなすので、その分（正解の `uart_lat_min_ms`）だけ Wi-Fi の遅れが短く出ます。

In [ ]:
out = run("trace_join.py --la synth/trace.bin --fs 2000000 --uart synth/uart_trace.csv "
          "--pcap synth/mqtt_ii5.pcap --db synth/db_ii5.csv --calib synth/calib.json")
ok(re.search(r"I2C のバイト = 送った JSON の adc (\d+)、送った JSON = DB の JSON \1、.* = Pico の temp_c \1", out) is not None,
   "全サンプルで、I2C のバイト = 送った JSON = DB の JSON、計算し直した温度 = Pico の temp_c")
wifi = num(r"publish〜Pi の eth0（Wi-Fi・中継機）\s+([\d.]+)", out)
print(f"Wi-Fi の遅れの中央値：解析 {wifi:.3f} ms、正解 {T_WIFI:.3f} ms、差 {T_WIFI - wifi:.3f} ms（受信の遅れの最小値 {T_UARTMIN:.3f} ms）")
ok(abs(T_WIFI - 8.4) < 0.1, f"正解の Wi-Fi の遅れの中央値は {T_WIFI:.2f} ms（本文の 8.4 ms 前後）")
ok(0.2 < T_WIFI - wifi < 0.45, "解析の中央値は正解より約 0.3 ms 短い（片側の誤差）")
s_med, m_sum = num(r"各層の中央値の和 ([\d.]+) ms", out), num(r"和の中央値 ([\d.]+) ms", out)
ok(abs(s_med - m_sum) > 0.1, f"各層の中央値の和 {s_med} ms ≠ 和の中央値 {m_sum} ms（分位点は足せない）")
show("trace.png")

最後に、「分位点は足せないが、平均は足せる」ことを、`trace_join.py` と同じ考え方で小さく確かめます（図II-5-10 の考え方。右に裾のある2つの遅れを足す）。値は図とは別の乱数なので、図の 9.3 ms・29.1 ms とは少し違います。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
a = 8.0 * np.exp(rng.normal(0, 0.5, 200_000))         # 中央値 8 ms の対数正規（裾が長い）
b = 1.0 * np.exp(rng.normal(0, 0.8, 200_000))         # 中央値 1 ms
s = a + b
print(f"中央値：{np.median(a):.2f} + {np.median(b):.2f} = {np.median(a) + np.median(b):.2f} ms、和の中央値 {np.median(s):.2f} ms")
print(f"95%点：{np.quantile(a, .95):.2f} + {np.quantile(b, .95):.2f} = {np.quantile(a, .95) + np.quantile(b, .95):.2f} ms、和の95%点 {np.quantile(s, .95):.2f} ms")
print(f"平均：{a.mean():.3f} + {b.mean():.3f} = {a.mean() + b.mean():.3f} ms、和の平均 {s.mean():.3f} ms")
ok(abs(np.median(s) - (np.median(a) + np.median(b))) > 0.1, "中央値は足し算どおりにならない")
print("平均だけは足し算どおりになる（データによらず、平均の性質としていつも成り立つ）")

## まとめ

- 合成の記録で、手順 A の「確かめること」（チップID 0x60、0x199F の一致、72 回のうち食い違い 1 回、注釈と一致 76 件、容量の範囲に 90 pF が入る、Wi-Fi の遅れが正解より約 0.3 ms 短い）と、II-5-7 の合成データの値（93.1 kHz、54 ns、80°・48 %）を確かめました。
- 実物の記録では、ロジックアナライザの水晶のずれ、プローブの容量、家の Wi-Fi で値が変わります。教科書の手順 B 以降で、同じプログラムを実物の記録（`data/`）に使ってください。
- このノートブックに入れていないもの：配線、Pi の UART と sigrok-cli での記録、Pico・Pi のプログラムの実行、tcpdump・DB・Grafana、PulseView での表示。